In [ ]:
!wget https://huggingface.co/TheBloke/OpenHermes-2.5-Mistral-7B-GGUF/resolve/main/openhermes-2.5-mistral-7b.Q4_K_M.gguf

In [ ]:
from torch import cuda

model_id = 'meta-llama/Llama-2-7b-chat-hf'
device = f'cuda:{cuda.current_device()}' if cuda.is_available() else 'cpu'

print(device)

In [ ]:
from torch import bfloat16
import transformers

# set quantization configuration to load large model with less GPU memory
# this requires the `bitsandbytes` library

bnb_config = transformers.BitsAndBytesConfig(
    load_in_4bit=True,  # 4-bit quantization
    bnb_4bit_quant_type='nf4',  # Normalized float 4
    bnb_4bit_use_double_quant=True,  # Second quantization after the first
    bnb_4bit_compute_dtype=bfloat16  # Computation type
)

In [ ]:
tokenizer = transformers.AutoTokenizer.from_pretrained(model_id)

# Llama 2 Model
model = transformers.AutoModelForCausalLM.from_pretrained(
    model_id,
    trust_remote_code=True,
    quantization_config=bnb_config,
    device_map='auto',
)
model.eval()

In [ ]:
# Our text generator
generator = transformers.pipeline(
    model=model, tokenizer=tokenizer,
    task='text-generation',
    temperature=0.1,
    max_new_tokens=500,
    repetition_penalty=1.1
)

In [ ]:
prompt = "Could you explain to me how 4-bit quantization works as if I am 5?"
res = generator(prompt)
print(res[0]["generated_text"])

In [ ]:
"""
<s>[INST] <<SYS>>

{{ System Prompt }}

<</SYS>>

{{ User Prompt }} [/INST]

{{ Model Answer }}
"""

In [ ]:
system_prompt = """
<s>[INST] <<SYS>>
You are a helpful, respectful and honest assistant for labeling topics.
<</SYS>>
"""

In [ ]:
example_prompt = """
I have a topic that contains the following documents:
- Traditional diets in most cultures were primarily plant-based with a little meat on top, but with the rise of industrial style meat production and factory farming, meat has become a staple food.
- Meat, but especially beef, is the word food in terms of emissions.
- Eating meat doesn't make you a bad person, not eating meat doesn't make you a good one.

The topic is described by the following keywords: 'meat, beef, eat, eating, emissions, steak, food, health, processed, chicken'.

Based on the information about the topic above, please create a short label of this topic. Make sure you to only return the label and nothing more.

[/INST] Environmental impacts of eating meat
"""

In [ ]:
main_prompt = """
[INST]
I have a topic that contains the following documents:
[DOCUMENTS]

The topic is described by the following keywords: '[KEYWORDS]'.

Based on the information about the topic above, please create a short label of this topic. Make sure you to only return the label and nothing more.
[/INST]
"""

In [ ]:
prompt = system_prompt + example_prompt + main_prompt

In [ ]:
from datasets import load_dataset

dataset = load_dataset("CShorten/ML-ArXiv-Papers")["train"]

# Extract abstracts to train on and corresponding titles
abstracts = dataset["abstract"]
titles = dataset["title"]

In [ ]:
# abstracts=["""  The dominant sequence transduction models are based on complex recurrent or
# convolutional neural networks in an encoder-decoder configuration. The best
# performing models also connect the encoder and decoder through an attention
# mechanism. We propose a new simple network architecture, the Transformer, based
# solely on attention mechanisms, dispensing with recurrence and convolutions
# entirely. Experiments on two machine translation tasks show these models to be
# superior in quality while being more parallelizable and requiring significantly
# less time to train. Our model achieves 28.4 BLEU on the WMT 2014
# English-to-German translation task, improving over the existing best results,
# including ensembles by over 2 BLEU. On the WMT 2014 English-to-French
# translation task, our model establishes a new single-model state-of-the-art
# BLEU score of 41.8 after training for 3.5 days on eight GPUs, a small fraction
# of the training costs of the best models from the literature. We show that the
# Transformer generalizes well to other tasks by applying it successfully to
# English constituency parsing both with large and limited training data."""]

In [ ]:
from sentence_transformers import SentenceTransformer

# Pre-calculate embeddings
embedding_model = SentenceTransformer("BAAI/bge-small-en")
embeddings = embedding_model.encode(abstracts, show_progress_bar=True)

In [ ]:
from umap import UMAP
from hdbscan import HDBSCAN

umap_model = UMAP(n_neighbors=15, n_components=5, min_dist=0.0, metric='cosine', random_state=42)
hdbscan_model = HDBSCAN(min_cluster_size=150, metric='euclidean', cluster_selection_method='eom', prediction_data=True)

In [ ]:
reduced_embeddings = UMAP(n_neighbors=15, n_components=2, min_dist=0.0, metric='cosine', random_state=42).fit_transform(embeddings)

In [ ]:
from bertopic.representation import KeyBERTInspired, MaximalMarginalRelevance, TextGeneration

# KeyBERT
keybert = KeyBERTInspired()

# MMR
mmr = MaximalMarginalRelevance(diversity=0.3)

# Text generation with Llama 2
llama2 = TextGeneration(generator, prompt=prompt)

# All representation models
representation_model = {
    "KeyBERT": keybert,
    "Llama2": llama2,
    "MMR": mmr,
}

In [ ]:
from bertopic import BERTopic

topic_model = BERTopic(

  # Sub-models
  embedding_model=embedding_model,
  umap_model=umap_model,
  hdbscan_model=hdbscan_model,
  representation_model=representation_model,

  # Hyperparameters
  top_n_words=10,
  verbose=True
)

# Train model
topics, probs = topic_model.fit_transform(abstracts, embeddings)

In [ ]:
topic_model.get_topic(1, full=True)["KeyBERT"]

In [ ]:
llama2_labels = [label[0][0].split("\n")[0] for label in topic_model.get_topics(full=True)["Llama2"].values()]
topic_model.set_topic_labels(llama2_labels)

In [ ]:
topic_model.visualize_documents(titles, reduced_embeddings=reduced_embeddings, hide_annotations=True, hide_document_hover=False, custom_labels=True)

In [ ]:
!pip install matplotlib

In [ ]:
!pip install seaborn

In [ ]:
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt

# Step 1: Prepare your F1 score data
# Rows: Models
# Columns: Tasks
heatmap_data = pd.DataFrame({
    "TBE-1": [0.42, 0.30, 0.61, 0.66],
    "TBE-2": [0.42, 0.54, 0.61, 0.66],
    "TBE-3": [0.42, 0.54, 0.61, 0.88]
}, index=[
    "Baseline (LIAR-RAW)",
    "Our Best (LIAR-RAW)",
    "Baseline (RAW-FC)",
    "Our Best (RAW-FC)"
])

# Step 2: Set up the plot
plt.figure(figsize=(10, 6))
sns.set(font_scale=1.0)

# Step 3: Create the heatmap
sns.heatmap(
    heatmap_data,
    annot=True,           # Show values
    cmap="YlGnBu",         # Color scheme
    fmt=".2f",             # Format the numbers
    linewidths=0.5,        # Line between cells
    linecolor='gray',
    cbar_kws={"label": "F1 Score"}
)

# Step 4: Labels and titles
plt.title("F1 Score Heatmap Across Tasks")
plt.xlabel("Task")
plt.ylabel("Model")

# Step 5: Save and show
plt.tight_layout()
plt.savefig("f1_score_heatmap.png")
plt.show()


In [ ]:
import matplotlib.pyplot as plt
import numpy as np

# X-axis labels (Dataset + Task)
x_labels = [
    "LIAR-RAW\nTBE-1",
    "LIAR-RAW\nTBE-2",
    "LIAR-RAW\nTBE-3",
    "RAW-FC\nTBE-1",
    "RAW-FC\nTBE-2",
    "RAW-FC\nTBE-3"
]

# Corresponding Macro F1 scores from your best models
f1_scores = [0.30, 0.54, 0.54, 0.66, 0.71, 0.88]

# Plotting
plt.figure(figsize=(10, 5))
plt.plot(f1_scores, marker='o', linestyle='dotted', color='blue', label='Macro F1 score')

# Annotate each point
for i, score in enumerate(f1_scores):
    plt.text(i, score + 0.02, f"{score:.2f}", ha='center', fontsize=10, color='blue')

# Formatting
plt.xticks(ticks=np.arange(len(x_labels)), labels=x_labels)
plt.ylim(0, 1.0)
plt.ylabel("Macro F1 Score")
plt.title("Macro F1 Performance Across Datasets and Tasks", fontsize=14, color='blue')
plt.grid(True, linestyle='--', alpha=0.5)
plt.legend(loc='lower right')
plt.tight_layout()
plt.show()


In [ ]:
import matplotlib.pyplot as plt
import numpy as np

# X-axis labels (task-dataset pairs)
x_labels = [
    "LIAR-RAW\nTBE-1",
    "LIAR-RAW\nTBE-2",
    "LIAR-RAW\nTBE-3",
    "RAW-FC\nTBE-1",
    "RAW-FC\nTBE-2",
    "RAW-FC\nTBE-3"
]

# Your best F1 scores
our_f1_scores = [0.30, 0.54, 0.54, 0.66, 0.71, 0.88]

# Baseline F1 scores
baseline_liar = 0.42
baseline_rawfc = 0.61
baseline_scores = [baseline_liar]*3 + [baseline_rawfc]*3

# Plotting
plt.figure(figsize=(10, 5))
plt.plot(our_f1_scores, marker='o', linestyle='dotted', color='blue', label='Our Best (Macro F1)')
plt.plot(baseline_scores, marker='x', linestyle='dashed', color='gray', label='Baseline (SOTA)')

# Annotate values
for i in range(len(x_labels)):
    plt.text(i, our_f1_scores[i] + 0.02, f"{our_f1_scores[i]:.2f}", ha='center', color='blue', fontsize=9)
    plt.text(i, baseline_scores[i] - 0.05, f"{baseline_scores[i]:.2f}", ha='center', color='gray', fontsize=9)

# Formatting
plt.xticks(ticks=np.arange(len(x_labels)), labels=x_labels)
plt.ylim(0, 1.0)
plt.ylabel("Macro F1 Score")
plt.title("Macro F1 Score per Task: Our Models vs Baseline", fontsize=13)
plt.grid(True, linestyle='--', alpha=0.5)
plt.legend(loc='lower right')
plt.tight_layout()
plt.show()


In [ ]:
import matplotlib.pyplot as plt
import numpy as np

# X-axis labels
x_labels = [
    "LIAR-RAW\nTBE-1",
    "LIAR-RAW\nTBE-2",
    "LIAR-RAW\nTBE-3",
    "RAW-FC\nTBE-1",
    "RAW-FC\nTBE-2",
    "RAW-FC\nTBE-3"
]

# Corrected F1 scores from your best models
our_f1_scores = [0.30, 0.32, 0.54, 0.66, 0.71, 0.88]

# Baseline F1 scores (SOTA references)
baseline_liar = [0.42, 0.42, 0.42]
baseline_rawfc = [0.61, 0.61, 0.61]
baseline_scores = baseline_liar + baseline_rawfc

# Plotting
plt.figure(figsize=(10, 5))
plt.plot(our_f1_scores, marker='o', linestyle='dotted', color='blue', label='Our Best (Macro F1)')
plt.plot(baseline_scores, marker='x', linestyle='dashed', color='gray', label='Baseline (SOTA)')

# Annotate values
for i in range(len(x_labels)):
    plt.text(i, our_f1_scores[i] + 0.02, f"{our_f1_scores[i]:.2f}", ha='center', color='blue', fontsize=9)
    plt.text(i, baseline_scores[i] - 0.05, f"{baseline_scores[i]:.2f}", ha='center', color='gray', fontsize=9)

# Formatting
plt.xticks(ticks=np.arange(len(x_labels)), labels=x_labels)
plt.ylim(0, 1.0)
plt.ylabel("Macro F1 Score")
plt.title("Macro F1 Score per Task: Our Models vs Baseline", fontsize=13)
plt.grid(True, linestyle='--', alpha=0.5)
plt.legend(loc='lower right')
plt.tight_layout()
plt.show()


In [ ]:
import matplotlib.pyplot as plt
import numpy as np

# Task-dataset labels
x_labels = [
    "LIAR-RAW\nTBE-1",
    "LIAR-RAW\nTBE-2",
    "LIAR-RAW\nTBE-3",
    "RAW-FC\nTBE-1",
    "RAW-FC\nTBE-2",
    "RAW-FC\nTBE-3"
]

# Your best F1 scores
our_f1_scores = [0.30, 0.32, 0.54, 0.66, 0.66, 0.88]

# Baseline F1 scores
baseline_scores = [0.42, 0.42, 0.42, 0.61, 0.61, 0.61]

# X-axis positions
x_liar = np.arange(3)
x_rawfc = np.arange(3, 6)

# Color settings
liar_color = '#1f77b4'    # Blue for LIAR-RAW
rawfc_color = '#2ca02c'   # Green for RAW-FC

# Start plotting
plt.figure(figsize=(10, 5))

# Our models
plt.plot(x_liar, [our_f1_scores[i] for i in x_liar], marker='o', markersize=10,
         linestyle='dotted', linewidth=2.5, color=liar_color, label='Our Best (LIAR-RAW)')
plt.plot(x_rawfc, [our_f1_scores[i] for i in x_rawfc], marker='o', markersize=10,
         linestyle='dotted', linewidth=2.5, color=rawfc_color, label='Our Best (RAW-FC)')

# Baseline models
plt.plot(x_liar, [baseline_scores[i] for i in x_liar], marker='x', markersize=10,
         linestyle='dashed', linewidth=2.5, color=liar_color, alpha=0.6, label='Baseline (LIAR-RAW)')
plt.plot(x_rawfc, [baseline_scores[i] for i in x_rawfc], marker='x', markersize=10,
         linestyle='dashed', linewidth=2.5, color=rawfc_color, alpha=0.6, label='Baseline (RAW-FC)')

# Annotate F1 values
for i in range(6):
    plt.text(i, our_f1_scores[i] + 0.02, f"{our_f1_scores[i]:.2f}", ha='center', fontsize=9, color='black')
    plt.text(i, baseline_scores[i] - 0.06, f"{baseline_scores[i]:.2f}", ha='center', fontsize=9, color='gray')

# Final plot settings
plt.xticks(ticks=np.arange(len(x_labels)), labels=x_labels)
plt.ylim(0, 1.0)
plt.ylabel("Macro F1 Score")
plt.title("Macro F1 Score per Task: Our Models vs Baselines", fontsize=14)
plt.grid(True, linestyle='--', alpha=0.5)
plt.legend(loc='lower right')
plt.tight_layout()
plt.show()


In [ ]:
import matplotlib.pyplot as plt
import numpy as np

# Task-dataset labels
x_labels = [
    "LIAR-RAW\nTBE-1",.36
    "LIAR-RAW\nTBE-2",
    "LIAR-RAW\nTBE-3",
    "RAW-FC\nTBE-1",
    "RAW-FC\nTBE-2",
    "RAW-FC\nTBE-3"
]

# F1 scores
our_f1_scores = [0.30, 0.32, 0.54, 0.66, 0.66, 0.88]
baseline_scores = [0.42, 0.42, 0.42, 0.61, 0.61, 0.61]

# X axis positions
x = np.arange(len(x_labels))
width = 0.35

# Color scheme
liar_color = '#1f77b4'   # Blue
rawfc_color = '#2ca02c'  # Green
baseline_color = '#aaaaaa'  # Gray

# Assign colors for bar categories
bar_colors = [liar_color]*3 + [rawfc_color]*3

# Create figure
plt.figure(figsize=(11, 6))

# Plot Our Best bars
plt.bar(x - width/2, our_f1_scores, width=width, color=bar_colors, label='Our Best (Macro F1)')

# Plot Baseline bars
plt.bar(x + width/2, baseline_scores, width=width, color=baseline_color, alpha=0.7, label='Baseline (SOTA)')

# Draw connecting lines (trends) for Our Best
plt.plot(x[:3] - width/2, our_f1_scores[:3], linestyle='-', linewidth=2.5, color=liar_color)
plt.plot(x[3:] - width/2, our_f1_scores[3:], linestyle='-', linewidth=2.5, color=rawfc_color)

# Draw connecting lines (trends) for Baseline
plt.plot(x[:3] + width/2, baseline_scores[:3], linestyle='--', linewidth=2.5, color=liar_color, alpha=0.6)
plt.plot(x[3:] + width/2, baseline_scores[3:], linestyle='--', linewidth=2.5, color=rawfc_color, alpha=0.6)

# Add circular markers at peak F1 scores
max_idx_our = np.argmax(our_f1_scores)
max_idx_base = np.argmax(baseline_scores)

plt.scatter(x[max_idx_our] - width/2, our_f1_scores[max_idx_our], s=150,
            facecolors='none', edgecolors='black', linewidth=2, label='Peak (Ours)')
plt.scatter(x[max_idx_base] + width/2, baseline_scores[max_idx_base], s=150,
            facecolors='none', edgecolors='darkgray', linewidth=2, label='Peak (Baseline)')

# Add F1 score annotations
for i in range(len(x)):
    plt.text(x[i] - width/2, our_f1_scores[i] + 0.02, f"{our_f1_scores[i]:.2f}", ha='center', fontsize=9)
    plt.text(x[i] + width/2, baseline_scores[i] + 0.02, f"{baseline_scores[i]:.2f}", ha='center', fontsize=9)

# Axis formatting
plt.xticks(ticks=x, labels=x_labels)
plt.ylim(0, 1.0)
plt.ylabel("Macro F1 Score")
plt.title("Task-wise Macro F1: Our Models vs Baselines (Trends & Peaks)", fontsize=14)
plt.grid(True, axis='y', linestyle='--', alpha=0.5)
plt.legend(loc='lower right')
plt.tight_layout()
plt.show()


In [ ]:
import matplotlib.pyplot as plt
import numpy as np

# Categories and Methods
categories = ['TBE-1', 'TBE-2', 'TBE-3']
methods = ['Our Best (LIAR-RAW)', 'Baseline (LIAR-RAW)',
           'Our Best (RAW-FC)', 'Baseline (RAW-FC)']

# Macro F1 scores for each method across tasks
data = [
    [0.30, 0.32, 0.54],  # Our Best (LIAR-RAW)
    [0.42, 0.42, 0.42],  # Baseline (LIAR-RAW)
    [0.66, 0.66, 0.88],  # Our Best (RAW-FC)
    [0.61, 0.61, 0.61],  # Baseline (RAW-FC)
]

# Bar styling
colors = ['#1f77b4', '#bbbbbb', '#2ca02c', '#999999']
hatches = ['/', 'x', '\\', '*']

# Setup bar locations
x = np.arange(len(categories))
total_width = 0.8
bar_width = total_width / len(methods).36

# Initialize figure
plt.figure(figsize=(10, 6))

# Plot bars
for i, (method, scores) in enumerate(zip(methods, data)):
    plt.bar(x + i * bar_width - total_width / 2,
            scores,
            width=bar_width,
            label=method,
            color=colors[i],
            hatch=hatches[i],
            edgecolor='black',
            linewidth=0.8)

# Axis and labels
plt.xticks(ticks=x, labels=categories)
plt.ylabel("Macro F1 Score")
plt.title("F1 Scores on LIAR-RAW and RAW-FC Across Tasks (TBE-1, 2, 3)")
plt.ylim(0, 1.05)
plt.grid(axis='y', linestyle='--', alpha=0.5)

# Legend below chart
plt.legend(loc='lower center', bbox_to_anchor=(0.5, -0.25),
           ncol=2, frameon=False)

# Layout adjustment
plt.tight_layout()
plt.show()


In [ ]:
import matplotlib.pyplot as plt
import numpy as np

# Categories (Tasks) and Methods
categories = ['TBE-1', 'TBE-2', 'TBE-3']
methods = ['Our Best (LIAR-RAW)', 'Baseline (LIAR-RAW)',
           'Our Best (RAW-FC)', 'Baseline (RAW-FC)'].36

# Macro F1 scores for each method across tasks
data = [
    [0.30, 0.32, 0.54],  # Our Best (LIAR-RAW)
    [0.42, 0.36, 0.36],  # Baseline (LIAR-RAW)
    [0.66, 0.66, 0.88],  # Our Best (RAW-FC)
    [0.61, 0.61, 0.61],  # Baseline (RAW-FC)
]

# Bar styles
colors = ['#1f77b4', '#bbbbbb', '#2ca02c', '#999999']
hatches = ['/', 'x', '\\', '*']

# X-axis setup
x = np.arange(len(categories))
total_width = 0.8
bar_width = total_width / len(methods)

# Plot setup
plt.figure(figsize=(10, 6))

# Draw each group of bars
for i, (method, scores) in enumerate(zip(methods, data)):
    positions = x + i * bar_width - total_width / 2.36
    bars = plt.bar(positions, scores, width=bar_width,
                   label=method, color=colors[i], hatch=hatches[i],
                   edgecolor='black', linewidth=0.8)

    # Annotate values above bars
    for pos, score in zip(positions, scores):
        plt.text(pos, score + 0.02, f"{score:.2f}", ha='center', va='bottom', fontsize=9)

# Axis and legend
plt.xticks(ticks=x, labels=categories)
plt.ylabel("Macro F1 Score")
plt.title("F1 Scores on LIAR-RAW and RAW-FC Across Tasks (TBE-1, 2, 3)")
plt.ylim(0, 1.05)
plt.grid(axis='y', linestyle='--', alpha=0.5)
plt.legend(loc='lower center', bbox_to_anchor=(0.5, -0.25),
           ncol=2, frameon=False)

plt.tight_layout()
plt.show()


In [ ]:
import matplotlib.pyplot as plt
import numpy as np

# Baseline F1 scores
baseline_liar_raw_MF1 = 0.42
baseline_rawfc_MF1 = 0.61

# Filtered F1 scores for models exceeding baseline or best within their TBE
f1_scores = {
    'LIAR-RAW': {
        'TBE-1': {'Lora-Llama': 0.30},  # highest in TBE-1
        'TBE-2': {'Lora-Mistral': 0.32},  # highest in TBE-2
        'TBE-3': {
            'RoBERTa-Llama': 0.52,
            'XLNet-Llama': 0.54,
            'RoBERTa-Gemma': 0.48,
            'XLNet-Qwen': 0.48
        }
    },
    'RAW-FC': {
        'TBE-1': {
            'Lora-Mistral': 0.65,
            'Lora-Llama': 0.65,
            'Lora-Qwen': 0.66,
            'Lora+-Llama': 0.65,
            'Lora+-Qwen': 0.65
        },
        'TBE-2': {
            'XLNet-Llama': 0.62,
            'Lora+-Llama': 0.71
        },
        'TBE-3': {
            'RoBERTa-Mistral': 0.83,
            'RoBERTa-Llama': 0.88,
            'RoBERTa-Qwen': 0.71,
            'RoBERTa-Falcon': 0.64,
            'XLNet-Mistral': 0.82,
            'XLNet-Llama': 0.87,
            'XLNet-Qwen': 0.70,
            'XLNet-Falcon': 0.74
        }
    }
}

# Function to generate annotated bar graphs
def plot_detailed_f1(dataset_name, baseline, f1_data):
    plt.figure(figsize=(14, 7))
    colors = {'TBE-1': '#6baed6', 'TBE-2': '#74c476', 'TBE-3': '#fd8d3c'}
    x_labels, heights, bar_colors = [], [], []

    # Flatten the data with TBE prefix
    for tbe, models in f1_data.items():
        for model, score in models.items():
            x_labels.append(f"{tbe}\n{model}")
            heights.append(score)
            bar_colors.append(colors[tbe])

    x = np.arange(len(x_labels))
    plt.bar(x, heights, color=bar_colors)
    plt.axhline(y=baseline, color='red', linestyle='--', linewidth=2, label='Baseline')

    # Annotate each bar
    for i, height in enumerate(heights):
        plt.text(i, height + 0.01, f"{height:.2f}", ha='center', va='bottom', fontsize=9)

    # Styling
    plt.xticks(x, x_labels, rotation=45, ha='right')
    plt.ylim(0, 1)
    plt.ylabel('Macro-F1 Score')
    plt.title(f'Filtered Macro-F1 Scores Above Baseline for {dataset_name}')
    plt.legend()
    plt.tight_layout()
    plt.grid(axis='y', linestyle='--', alpha=0.6)
    plt.show()

# Plot for both datasets
plot_detailed_f1('LIAR-RAW', baseline_liar_raw_MF1, f1_scores['LIAR-RAW'])
plot_detailed_f1('RAW-FC', baseline_rawfc_MF1, f1_scores['RAW-FC'])


In [ ]:
import matplotlib.pyplot as plt
import numpy as np

# Baseline F1 scores
baseline_liar_raw_MF1 = 0.42
baseline_rawfc_MF1 = 0.61

# Filtered F1 scores
f1_scores = {
    'LIAR-RAW': {
        'TBE-1': {'Lora-Llama': 0.30},
        'TBE-2': {'Lora-Mistral': 0.32},
        'TBE-3': {
            'RoBERTa-Llama': 0.52,
            'XLNet-Llama': 0.54,
            'RoBERTa-Gemma': 0.48,
            'XLNet-Qwen': 0.48
        }
    },
    'RAW-FC': {
        'TBE-1': {
            'Lora-Mistral': 0.65,
            'Lora-Llama': 0.65,
            'Lora-Qwen': 0.66,
            'Lora+-Llama': 0.65,
            'Lora+-Qwen': 0.65
        },
        'TBE-2': {
            'XLNet-Llama': 0.62,
            'Lora+-Llama': 0.71
        },
        'TBE-3': {
            'RoBERTa-Mistral': 0.83,
            'RoBERTa-Llama': 0.88,
            'RoBERTa-Qwen': 0.71,
            'RoBERTa-Falcon': 0.64,
            'XLNet-Mistral': 0.82,
            'XLNet-Llama': 0.87,
            'XLNet-Qwen': 0.70,
            'XLNet-Falcon': 0.74
        }
    }
}

# Plot function
def plot_clean_f1(dataset_name, baseline, f1_data):
    plt.figure(figsize=(14, 7))
    colors = {'TBE-1': '#6baed6', 'TBE-2': '#74c476', 'TBE-3': '#fd8d3c'}
    
    x_labels, heights, bar_colors, legend_labels = [], [], [], []

    for tbe, models in f1_data.items():
        for model, score in models.items():
            x_labels.append(model)
            heights.append(score)
            bar_colors.append(colors[tbe])
            legend_labels.append(tbe)

    x = np.arange(len(x_labels))
    bars = plt.bar(x, heights, color=bar_colors)

    # Add legend only once per TBE color
    from matplotlib.patches import Patch
    legend_elements = [
        Patch(facecolor='#6baed6', label='TBE-1'),
        Patch(facecolor='#74c476', label='TBE-2'),
        Patch(facecolor='#fd8d3c', label='TBE-3'),
        Patch(facecolor='red', label='Baseline')
    ]

    # Plot baseline
    plt.axhline(y=baseline, color='red', linestyle='--', linewidth=2)

    # Annotate bars
    for i, height in enumerate(heights):
        plt.text(i, height + 0.01, f"{height:.2f}", ha='center', va='bottom', fontsize=9)

    plt.xticks(x, x_labels, rotation=45, ha='right')
    plt.ylim(0, 1)
    plt.ylabel('Macro-F1 Score')
    plt.title(f'Macro-F1 Scores Above Baseline for {dataset_name}')
    plt.legend(handles=legend_elements)
    plt.grid(axis='y', linestyle='--', alpha=0.6)
    plt.tight_layout()
    plt.show()

# Generate the plots
plot_clean_f1('LIAR-RAW', baseline_liar_raw_MF1, f1_scores['LIAR-RAW'])
plot_clean_f1('RAW-FC', baseline_rawfc_MF1, f1_scores['RAW-FC'])


In [ ]:
import matplotlib.pyplot as plt
import numpy as np
from matplotlib.lines import Line2D
from matplotlib.patches import Patch

# Baseline F1 scores
baseline_liar_raw_MF1 = 0.42
baseline_rawfc_MF1 = 0.61

# Filtered F1 scores
f1_scores = {
    'LIAR-RAW': {
        'TBE-1': {'Lora-Llama': 0.30},
        'TBE-2': {'Lora-Mistral': 0.32},
        'TBE-3': {
            'RoBERTa-Llama': 0.52,
            'XLNet-Llama': 0.54,
            'RoBERTa-Gemma': 0.48,
            'XLNet-Qwen': 0.48
        }
    },
    'RAW-FC': {
        'TBE-1': {
            'Lora-Mistral': 0.65,
            'Lora-Llama': 0.65,
            'Lora-Qwen': 0.66,
            'Lora+-Llama': 0.65,
            'Lora+-Qwen': 0.65
        },
        'TBE-2': {
            'XLNet-Llama': 0.62,
            'Lora+-Llama': 0.71
        },
        'TBE-3': {
            'RoBERTa-Mistral': 0.83,
            'RoBERTa-Llama': 0.88,
            'RoBERTa-Qwen': 0.71,
            'RoBERTa-Falcon': 0.64,
            'XLNet-Mistral': 0.82,
            'XLNet-Llama': 0.87,
            'XLNet-Qwen': 0.70,
            'XLNet-Falcon': 0.74
        }
    }
}

# Plot function
def plot_clean_f1(dataset_name, baseline, f1_data):
    plt.figure(figsize=(14, 7))
    colors = {'TBE-1': '#6baed6', 'TBE-2': '#74c476', 'TBE-3': '#fd8d3c'}
    
    x_labels, heights, bar_colors = [], [], []

    for tbe, models in f1_data.items():
        for model, score in models.items():
            x_labels.append(model)
            heights.append(score)
            bar_colors.append(colors[tbe])

    x = np.arange(len(x_labels))
    plt.bar(x, heights, color=bar_colors)

    # Plot the baseline as a dashed line
    plt.axhline(y=baseline, color='red', linestyle='--', linewidth=2)

    # Annotate each bar
    for i, height in enumerate(heights):
        plt.text(i, height + 0.01, f"{height:.2f}", ha='center', va='bottom', fontsize=9)

    plt.xticks(x, x_labels, rotation=45, ha='right')
    plt.ylim(0, 1)
    plt.ylabel('Macro-F1 Score')
    plt.title(f'Macro-F1 Scores Above Baseline for {dataset_name}')
    plt.grid(axis='y', linestyle='--', alpha=0.6)
    plt.tight_layout()

    # Correct legend
    legend_elements = [
        Patch(facecolor='#6baed6', label='TBE-1'),
        Patch(facecolor='#74c476', label='TBE-2'),
        Patch(facecolor='#fd8d3c', label='TBE-3'),
        Line2D([0], [0], color='red', linestyle='--', linewidth=2, label='Baseline')
    ]
    plt.legend(handles=legend_elements)
    plt.show()

# Generate the corrected plots
plot_clean_f1('LIAR-RAW', baseline_liar_raw_MF1, f1_scores['LIAR-RAW'])
plot_clean_f1('RAW-FC', baseline_rawfc_MF1, f1_scores['RAW-FC'])


In [ ]:
import matplotlib.pyplot as plt
import numpy as np
from matplotlib.lines import Line2D
from matplotlib.patches import Patch

# Baseline values
baseline_liar_raw_MF1 = 0.42
baseline_rawfc_MF1 = 0.61

# Filtered F1 values for LAIR-RAW and RAW-FC
f1_scores_updated = {
    'LIAR-RAW': {
        'IBE-1': {'Mistral': 0.22},
        'IBE-2': {'Llama Llama': 0.22},
        'IBE-3': {'Mistral, Llama, Qwen, Falcon': 0.21},
        'IBE-4': {'Mistral and Falcon': 0.14},
        'TBE-1': {'Lora-Llama': 0.30},
        'TBE-2': {'Lora-Mistral': 0.32},
        'TBE-3': {
            'RoBERTa-Llama': 0.52,
            'XLNet-Llama': 0.54,
            'RoBERTa-Gemma': 0.48,
            'XLNet-Qwen': 0.48
        }
    },
    'RAW-FC': {
        'IBE-1': {'Qwen': 0.59},
        'IBE-2': {'Llama': 0.62},
        'IBE-3': {'Qwen': 0.52},
        'IBE-4': {'Qwen and Mistral': 0.43},
        'TBE-1': {
            'Lora-Mistral': 0.65,
            'Lora-Llama': 0.65,
            'Lora-Qwen': 0.66,
            'Lora+-Llama': 0.65,
            'Lora+-Qwen': 0.65
        },
        'TBE-2': {
            'XLNet-Llama': 0.62,
            'Lora+-Llama': 0.71
        },
        'TBE-3': {
            'RoBERTa-Mistral': 0.83,
            'RoBERTa-Llama': 0.88,
            'RoBERTa-Qwen': 0.71,
            'RoBERTa-Falcon': 0.64,
            'XLNet-Mistral': 0.82,
            'XLNet-Llama': 0.87,
            'XLNet-Qwen': 0.70,
            'XLNet-Falcon': 0.74
        }
    }
}

# Group color coding
group_colors = {
    'IBE-1': '#fbb4ae',
    'IBE-2': '#b3cde3',
    'IBE-3': '#ccebc5',
    'IBE-4': '#decbe4',
    'TBE-1': '#6baed6',
    'TBE-2': '#74c476',
    'TBE-3': '#fd8d3c'
}

# Plotting function
def plot_final_f1_compact(dataset_name, baseline, f1_data):
    plt.figure(figsize=(15, 8))

    x_labels, heights, bar_colors = [], [], []

    for group, models in f1_data.items():
        for model, score in models.items():
            x_labels.append(model)
            heights.append(score)
            bar_colors.append(group_colors[group])

    # Create tighter x positions
    x = np.arange(len(x_labels)) * 0.7
    bar_width = 0.35

    # Draw bars
    plt.bar(x, heights, color=bar_colors, width=bar_width)

    # Draw baseline line and label
    plt.axhline(y=baseline, color='red', linestyle='--', linewidth=2)
    plt.text(x[-1], baseline + 0.01, f'Baseline = {baseline:.2f}', color='red',
             ha='right', va='bottom', fontsize=11, fontweight='bold')

    # Bar labels
    for i, height in enumerate(heights):
        plt.text(x[i], height + 0.01, f"{height:.2f}", ha='center', va='bottom', fontsize=9)

    plt.xticks(x, x_labels, rotation=45, ha='right')
    plt.ylim(0, 1)
    plt.ylabel('Macro-F1 Score')
    plt.title(f'Macro-F1 Scores Including IBE and TBE for {dataset_name}')
    plt.grid(axis='y', linestyle='--', alpha=0.6)
    plt.tight_layout()

    # Custom legend
    legend_elements = [Patch(facecolor=color, label=group) for group, color in group_colors.items()]
    legend_elements.append(Line2D([0], [0], color='red', linestyle='--', linewidth=2, label='Baseline'))
    plt.legend(handles=legend_elements)
    plt.show()

# Plot for both datasets
plot_final_f1_compact('LIAR-RAW', baseline_liar_raw_MF1, f1_scores_updated['LIAR-RAW'])
plot_final_f1_compact('RAW-FC', baseline_rawfc_MF1, f1_scores_updated['RAW-FC'])


In [ ]:
import matplotlib.pyplot as plt
import numpy as np
from matplotlib.lines import Line2D
from matplotlib.patches import Patch

# Baseline for LIAR-RAW
baseline_liar_raw_MF1 = 0.42

# Filtered F1 scores for LIAR-RAW
f1_scores_liar_raw = {
    'IBE-1': {'Mistral': 0.22},
    'IBE-2': {'Llama Llama': 0.22},
    'IBE-3': {'Mistral, Llama, Qwen, Falcon': 0.21},
    'IBE-4': {'Mistral and Falcon': 0.14},
    'TBE-1': {'Lora-Llama': 0.30},
    'TBE-2': {'Lora-Mistral': 0.32},
    'TBE-3': {
        'RoBERTa-Llama': 0.52,
        'XLNet-Llama': 0.54,
        'RoBERTa-Gemma': 0.48,
        'XLNet-Qwen': 0.48
    }
}

# Color codes
group_colors = {
    'IBE-1': '#fbb4ae',
    'IBE-2': '#b3cde3',
    'IBE-3': '#ccebc5',
    'IBE-4': '#decbe4',
    'TBE-1': '#6baed6',
    'TBE-2': '#74c476',
    'TBE-3': '#fd8d3c'
}

# Plot function
def plot_liar_raw_f1():
    plt.figure(figsize=(15, 8))
    x_labels, heights, bar_colors = [], [], []

    for group, models in f1_scores_liar_raw.items():
        for model, score in models.items():
            x_labels.append(model)
            heights.append(score)
            bar_colors.append(group_colors[group])

    x = np.arange(len(x_labels)) * 0.2  # ✅ reduced spacing between bars
    bar_width = 0.1  # ✅ consistent thin bar

    plt.bar(x, heights, color=bar_colors, width=bar_width)

    # ✅ Baseline line only (no value)
    plt.axhline(y=baseline_liar_raw_MF1, color='red', linestyle='--', linewidth=2)

    # Annotate F1 score on each bar
    for i, height in enumerate(heights):
        plt.text(x[i], height + 0.01, f"{height:.2f}", ha='center', va='bottom', fontsize=9)

    plt.xticks(x, x_labels, rotation=45, ha='right')
    plt.ylim(0, 1)
    plt.ylabel('Macro-F1 Score')
    plt.title('Macro-F1 Scores Including IBE and TBE for LIAR-RAW')
    plt.grid(axis='y', linestyle='--', alpha=0.6)
    plt.tight_layout()

    # Custom legend
    legend_elements = [Patch(facecolor=color, label=group) for group, color in group_colors.items()]
    legend_elements.append(Line2D([0], [0], color='red', linestyle='--', linewidth=2, label='Baseline'))
    plt.legend(handles=legend_elements)
    plt.show()

# Call the plot
plot_liar_raw_f1()


In [ ]:
import matplotlib.pyplot as plt
import numpy as np
from matplotlib.lines import Line2D
from matplotlib.patches import Patch

# Baseline for RAW-FC
baseline_rawfc_MF1 = 0.61

# Filtered F1 scores for RAW-FC
f1_scores_rawfc = {
    'IBE-1': {'Qwen': 0.59},
    'IBE-2': {'Llama': 0.62},
    'IBE-3': {'Qwen': 0.52},
    'IBE-4': {'Qwen and Mistral': 0.43},
    'TBE-1': {
        'Lora-Mistral': 0.65,
        'Lora-Llama': 0.65,
        'Lora-Qwen': 0.66,
        'Lora+-Llama': 0.65,
        'Lora+-Qwen': 0.65
    },
    'TBE-2': {
        'XLNet-Llama': 0.62,
        'Lora+-Llama': 0.71
    },
    'TBE-3': {
        'RoBERTa-Mistral': 0.83,
        'RoBERTa-Llama': 0.88,
        'RoBERTa-Qwen': 0.71,
        'RoBERTa-Falcon': 0.64,
        'XLNet-Mistral': 0.82,
        'XLNet-Llama': 0.87,
        'XLNet-Qwen': 0.70,
        'XLNet-Falcon': 0.74
    }
}

# Color codes
group_colors = {
    'IBE-1': '#fbb4ae',
    'IBE-2': '#b3cde3',
    'IBE-3': '#ccebc5',
    'IBE-4': '#decbe4',
    'TBE-1': '#6baed6',
    'TBE-2': '#74c476',
    'TBE-3': '#fd8d3c'
}

# Plot function
def plot_rawfc_f1():
    plt.figure(figsize=(15, 8))
    x_labels, heights, bar_colors = [], [], []

    for group, models in f1_scores_rawfc.items():
        for model, score in models.items():
            x_labels.append(model)
            heights.append(score)
            bar_colors.append(group_colors[group])

    x = np.arange(len(x_labels)) * 0.7
    bar_width = 0.35

    plt.bar(x, heights, color=bar_colors, width=bar_width)
    plt.axhline(y=baseline_rawfc_MF1, color='red', linestyle='--', linewidth=2)
    plt.text(x[-1], baseline_rawfc_MF1 + 0.01, f'Baseline = {baseline_rawfc_MF1:.2f}', color='red',
             ha='right', va='bottom', fontsize=11, fontweight='bold')

    for i, height in enumerate(heights):
        plt.text(x[i], height + 0.01, f"{height:.2f}", ha='center', va='bottom', fontsize=9)

    plt.xticks(x, x_labels, rotation=45, ha='right')
    plt.ylim(0, 1)
    plt.ylabel('Macro-F1 Score')
    plt.title('Macro-F1 Scores Including IBE and TBE for RAW-FC')
    plt.grid(axis='y', linestyle='--', alpha=0.6)
    plt.tight_layout()

    legend_elements = [Patch(facecolor=color, label=group) for group, color in group_colors.items()]
    legend_elements.append(Line2D([0], [0], color='red', linestyle='--', linewidth=2, label='Baseline'))
    plt.legend(handles=legend_elements)
    plt.show()

# Call the plot
plot_rawfc_f1()


In [ ]:
import matplotlib.pyplot as plt
import numpy as np
from matplotlib.lines import Line2D
from matplotlib.patches import Patch

# Common settings
bar_width = 0.15
bar_spacing = 0.4  # reduce space between bars

group_colors = {
    'IBE-1': '#fbb4ae',
    'IBE-2': '#b3cde3',
    'IBE-3': '#ccebc5',
    'IBE-4': '#decbe4',
    'TBE-1': '#6baed6',
    'TBE-2': '#74c476',
    'TBE-3': '#fd8d3c'
}

# Dataset 1: LIAR-RAW
baseline_liar_raw = 0.42
f1_scores_liar_raw = {
    'IBE-1': {'Mistral': 0.22},
    'IBE-2': {'Llama Llama': 0.22},
    'IBE-3': {'Mistral, Llama, Qwen, Falcon': 0.21},
    'IBE-4': {'Mistral and Falcon': 0.14},
    'TBE-1': {'Lora-Llama': 0.30},
    'TBE-2': {'Lora-Mistral': 0.32},
    'TBE-3': {
        'RoBERTa-Llama': 0.52,
        'XLNet-Llama': 0.54,
        'RoBERTa-Gemma': 0.48,
        'XLNet-Qwen': 0.48
    }
}

# Dataset 2: RAW-FC
baseline_rawfc = 0.61
f1_scores_rawfc = {
    'IBE-1': {'Qwen': 0.59},
    'IBE-2': {'Llama': 0.62},
    'IBE-3': {'Qwen': 0.52},
    'IBE-4': {'Qwen and Mistral': 0.43},
    'TBE-1': {
        'Lora-Mistral': 0.65,
        'Lora-Llama': 0.65,
        'Lora-Qwen': 0.66,
        'Lora+-Llama': 0.65,
        'Lora+-Qwen': 0.65
    },
    'TBE-2': {
        'XLNet-Llama': 0.62,
        'Lora+-Llama': 0.71
    },
    'TBE-3': {
        'RoBERTa-Mistral': 0.83,
        'RoBERTa-Llama': 0.88,
        'RoBERTa-Qwen': 0.71,
        'RoBERTa-Falcon': 0.64,
        'XLNet-Mistral': 0.82,
        'XLNet-Llama': 0.87,
        'XLNet-Qwen': 0.70,
        'XLNet-Falcon': 0.74
    }
}

# Plotting function
def plot_f1_scores(dataset_name, f1_data, baseline_value):
    plt.figure(figsize=(15, 8))

    x_labels, heights, colors = [], [], []

    for group, models in f1_data.items():
        for model, score in models.items():
            x_labels.append(model)
            heights.append(score)
            colors.append(group_colors[group])

    x = np.arange(len(x_labels)) * bar_spacing
    plt.bar(x, heights, color=colors, width=bar_width)

    # Baseline line only (no text)
    plt.axhline(y=baseline_value, color='red', linestyle='--', linewidth=2)

    # Annotate each bar
    for i, height in enumerate(heights):
        plt.text(x[i], height + 0.01, f"{height:.2f}", ha='center', va='bottom', fontsize=9)

    plt.xticks(x, x_labels, rotation=45, ha='right')
    plt.ylim(0, 1)
    plt.ylabel('Macro-F1 Score')
    plt.title(f'Macro-F1 Scores Including IBE and TBE for {dataset_name}')
    plt.grid(axis='y', linestyle='--', alpha=0.6)
    plt.tight_layout()

    # Legend
    legend_elements = [Patch(facecolor=color, label=group) for group, color in group_colors.items()]
    legend_elements.append(Line2D([0], [0], color='red', linestyle='--', linewidth=2, label='Baseline'))
    plt.legend(handles=legend_elements)
    plt.show()

# Plot for LIAR-RAW
plot_f1_scores('LIAR-RAW', f1_scores_liar_raw, baseline_liar_raw)

# Plot for RAW-FC
plot_f1_scores('RAW-FC', f1_scores_rawfc, baseline_rawfc)


In [ ]:
import matplotlib.pyplot as plt
import numpy as np
from matplotlib.lines import Line2D
from matplotlib.patches import Patch

# Common bar settings
bar_width = 0.15
bar_spacing = 0.4

group_colors = {
    'IBE-1': '#fbb4ae',
    'IBE-2': '#b3cde3',
    'IBE-3': '#ccebc5',
    'IBE-4': '#decbe4',
    'TBE-1': '#6baed6',
    'TBE-2': '#74c476',
    'TBE-3': '#fd8d3c'
}

# LIAR-RAW data
baseline_liar_raw = 0.42
f1_scores_liar_raw = {
    'IBE-1': {'Mistral': 0.22},
    'IBE-2': {'Llama Llama': 0.22},
    'IBE-3': {'Mistral, Llama, Qwen, Falcon': 0.21},
    'IBE-4': {'Mistral and Falcon': 0.14},
    'TBE-1': {'Lora-Llama': 0.30},
    'TBE-2': {'Lora-Mistral': 0.32},
    'TBE-3': {
        'RoBERTa-Llama': 0.52,
        'XLNet-Llama': 0.54,
        'RoBERTa-Gemma': 0.48,
        'XLNet-Qwen': 0.48
    }
}

# RAW-FC data
baseline_rawfc = 0.61
f1_scores_rawfc = {
    'IBE-1': {'Qwen': 0.59},
    'IBE-2': {'Llama': 0.62},
    'IBE-3': {'Qwen': 0.52},
    'IBE-4': {'Qwen and Mistral': 0.43},
    'TBE-1': {
        'Lora-Mistral': 0.65,
        'Lora-Llama': 0.65,
        'Lora-Qwen': 0.66,
        'Lora+-Llama': 0.65,
        'Lora+-Qwen': 0.65
    },
    'TBE-2': {
        'XLNet-Llama': 0.62,
        'Lora+-Llama': 0.71
    },
    'TBE-3': {
        'RoBERTa-Mistral': 0.83,
        'RoBERTa-Llama': 0.88,
        'RoBERTa-Qwen': 0.71,
        'RoBERTa-Falcon': 0.64,
        'XLNet-Mistral': 0.82,
        'XLNet-Llama': 0.87,
        'XLNet-Qwen': 0.70,
        'XLNet-Falcon': 0.74
    }
}

# Unified plotting function with dynamic width
def plot_f1_scores(dataset_name, f1_data, baseline_value):
    num_bars = sum(len(models) for models in f1_data.values())
    fig_width = max(12, num_bars * 0.6)  # adjust figure width based on number of bars

    plt.figure(figsize=(fig_width, 8))

    x_labels, heights, colors = [], [], []

    for group, models in f1_data.items():
        for model, score in models.items():
            x_labels.append(model)
            heights.append(score)
            colors.append(group_colors[group])

    x = np.arange(len(x_labels)) * bar_spacing
    plt.bar(x, heights, color=colors, width=bar_width)

    # Draw baseline
    plt.axhline(y=baseline_value, color='red', linestyle='--', linewidth=2)

    # Annotate bars
    for i, height in enumerate(heights):
        plt.text(x[i], height + 0.01, f"{height:.2f}", ha='center', va='bottom', fontsize=9)

    plt.xticks(x, x_labels, rotation=45, ha='right')
    plt.ylim(0, 1)
    plt.ylabel('Macro-F1 Score')
    plt.title(f'Macro-F1 Scores Including IBE and TBE for {dataset_name}')
    plt.grid(axis='y', linestyle='--', alpha=0.6)
    plt.tight_layout()

    # Legend
    legend_elements = [Patch(facecolor=color, label=group) for group, color in group_colors.items()]
    legend_elements.append(Line2D([0], [0], color='red', linestyle='--', linewidth=2, label='Baseline'))
    plt.legend(handles=legend_elements)
    plt.show()

# Plot both datasets
plot_f1_scores('LIAR-RAW', f1_scores_liar_raw, baseline_liar_raw)
plot_f1_scores('RAW-FC', f1_scores_rawfc, baseline_rawfc)


In [ ]:
import matplotlib.pyplot as plt
import numpy as np
from matplotlib.lines import Line2D
from matplotlib.patches import Patch

# Common bar settings
bar_width = 0.15
bar_spacing = 0.3  # reduced space between bars

# Group colors
group_colors = {
    'IBE-1': '#fbb4ae',
    'IBE-2': '#b3cde3',
    'IBE-3': '#ccebc5',
    'IBE-4': '#decbe4',
    'TBE-1': '#6baed6',
    'TBE-2': '#74c476',
    'TBE-3': '#fd8d3c'
}

# LIAR-RAW data
baseline_liar_raw = 0.42
f1_scores_liar_raw = {
    'IBE-1': {'Mistral': 0.22},
    'IBE-2': {'Llama Llama': 0.22},
    'IBE-3': {'Mistral, Llama, Qwen, Falcon': 0.21},
    'IBE-4': {'Mistral and Falcon': 0.14},
    'TBE-1': {'Lora-Llama': 0.30},
    'TBE-2': {'Lora-Mistral': 0.32},
    'TBE-3': {
        'RoBERTa-Llama': 0.52,
        'XLNet-Llama': 0.54,
        'RoBERTa-Gemma': 0.48,
        'XLNet-Qwen': 0.48
    }
}

# RAW-FC data
baseline_rawfc = 0.61
f1_scores_rawfc = {
    'IBE-1': {'Qwen': 0.59},
    'IBE-2': {'Llama': 0.62},
    'IBE-3': {'Qwen': 0.52},
    'IBE-4': {'Qwen and Mistral': 0.43},
    'TBE-1': {
        'Lora-Mistral': 0.65,
        'Lora-Llama': 0.65,
        'Lora-Qwen': 0.66,
        'Lora+-Llama': 0.65,
        'Lora+-Qwen': 0.65
    },
    'TBE-2': {
        'XLNet-Llama': 0.62,
        'Lora+-Llama': 0.71
    },
    'TBE-3': {
        'RoBERTa-Mistral': 0.83,
        'RoBERTa-Llama': 0.88,
        'RoBERTa-Qwen': 0.71,
        'RoBERTa-Falcon': 0.64,
        'XLNet-Mistral': 0.82,
        'XLNet-Llama': 0.87,
        'XLNet-Qwen': 0.70,
        'XLNet-Falcon': 0.74
    }
}

# Plotting function
def plot_f1_scores(dataset_name, f1_data, baseline_value):
    num_bars = sum(len(models) for models in f1_data.values())
    fig_width = max(12, num_bars * 0.55)  # auto-scale based on bar count

    plt.figure(figsize=(fig_width, 8))

    x_labels, heights, colors = [], [], []

    for group, models in f1_data.items():
        for model, score in models.items():
            x_labels.append(model)
            heights.append(score)
            colors.append(group_colors[group])

    x = np.arange(len(x_labels)) * bar_spacing
    plt.bar(x, heights, color=colors, width=bar_width)

    # Baseline line only (no value label)
    plt.axhline(y=baseline_value, color='red', linestyle='--', linewidth=2)

    # Add F1 score labels
    for i, height in enumerate(heights):
        plt.text(x[i], height + 0.01, f"{height:.2f}", ha='center', va='bottom', fontsize=9)

    # Formatting
    plt.xticks(x, x_labels, rotation=45, ha='right')
    plt.ylim(0, 1)
    plt.ylabel('Macro-F1 Score')
    plt.title(f'Macro-F1 Scores Including IBE and TBE for {dataset_name}')
    plt.grid(axis='y', linestyle='--', alpha=0.6)
    plt.tight_layout()

    # Legend
    legend_elements = [Patch(facecolor=color, label=group) for group, color in group_colors.items()]
    legend_elements.append(Line2D([0], [0], color='red', linestyle='--', linewidth=2, label='Baseline'))
    plt.legend(handles=legend_elements)
    plt.show()

# Plot for LIAR-RAW
plot_f1_scores('LIAR-RAW', f1_scores_liar_raw, baseline_liar_raw)

# Plot for RAW-FC
plot_f1_scores('RAW-FC', f1_scores_rawfc, baseline_rawfc)


In [ ]:
import matplotlib.pyplot as plt
import numpy as np
from matplotlib.lines import Line2D
from matplotlib.patches import Patch

# Common bar settings
bar_width = 0.15
bar_spacing = 0.3

# Group colors
group_colors = {
    'IBE-1': '#fbb4ae',
    'IBE-2': '#b3cde3',
    'IBE-3': '#ccebc5',
    'IBE-4': '#decbe4',
    'TBE-1': '#6baed6',
    'TBE-2': '#74c476',
    'TBE-3': '#fd8d3c'
}

# LIAR-RAW dataset
baseline_liar_raw = 0.42
f1_scores_liar_raw = {
    'IBE-1': {'Mistral': 0.22},
    'IBE-2': {'Llama Llama': 0.22},
    'IBE-3': {'Mistral, Llama, Qwen, Falcon': 0.21},
    'IBE-4': {'Mistral and Falcon': 0.14},
    'TBE-1': {'Lora-Llama': 0.30},
    'TBE-2': {'Lora-Mistral': 0.32},
    'TBE-3': {
        'RoBERTa-Llama': 0.52,
        'XLNet-Llama': 0.54,
        'RoBERTa-Gemma': 0.48,
        'XLNet-Qwen': 0.48
    }
}

# RAW-FC dataset
baseline_rawfc = 0.61
f1_scores_rawfc = {
    'IBE-1': {'Qwen': 0.59},
    'IBE-2': {'Llama': 0.62},
    'IBE-3': {'Qwen': 0.52},
    'IBE-4': {'Qwen and Mistral': 0.43},
    'TBE-1': {
        'Lora-Mistral': 0.65,
        'Lora-Llama': 0.65,
        'Lora-Qwen': 0.66,
        'Lora+-Llama': 0.65,
        'Lora+-Qwen': 0.65
    },
    'TBE-2': {
        'XLNet-Llama': 0.62,
        'Lora+-Llama': 0.71
    },
    'TBE-3': {
        'RoBERTa-Mistral': 0.83,
        'RoBERTa-Llama': 0.88,
        'RoBERTa-Qwen': 0.71,
        'RoBERTa-Falcon': 0.64,
        'XLNet-Mistral': 0.82,
        'XLNet-Llama': 0.87,
        'XLNet-Qwen': 0.70,
        'XLNet-Falcon': 0.74
    }
}

# Plotting function with baseline label in legend
def plot_f1_scores(dataset_name, f1_data, baseline_value):
    num_bars = sum(len(models) for models in f1_data.values())
    fig_width = max(12, num_bars * 0.55)

    plt.figure(figsize=(fig_width, 8))

    x_labels, heights, colors = [], [], []

    for group, models in f1_data.items():
        for model, score in models.items():
            x_labels.append(model)
            heights.append(score)
            colors.append(group_colors[group])

    x = np.arange(len(x_labels)) * bar_spacing
    plt.bar(x, heights, color=colors, width=bar_width)

    # Draw baseline line
    plt.axhline(y=baseline_value, color='red', linestyle='--', linewidth=2)

    # Bar value annotations
    for i, height in enumerate(heights):
        plt.text(x[i], height + 0.01, f"{height:.2f}", ha='center', va='bottom', fontsize=9)

    plt.xticks(x, x_labels, rotation=45, ha='right')
    plt.ylim(0, 1)
    plt.ylabel('Macro-F1 Score')
    plt.title(f'Macro-F1 Scores Including IBE and TBE for {dataset_name}')
    plt.grid(axis='y', linestyle='--', alpha=0.6)
    plt.tight_layout()

    # Legend with baseline value
    legend_elements = [Patch(facecolor=color, label=group) for group, color in group_colors.items()]
    legend_elements.append(Line2D([0], [0], color='red', linestyle='--', linewidth=2,
                                  label=f'Baseline = {baseline_value:.2f}'))
    plt.legend(handles=legend_elements)
    plt.show()

# Plot LIAR-RAW
plot_f1_scores('LIAR-RAW', f1_scores_liar_raw, baseline_liar_raw)

# Plot RAW-FC
plot_f1_scores('RAW-FC', f1_scores_rawfc, baseline_rawfc)


In [ ]:
import matplotlib.pyplot as plt
import numpy as np
from matplotlib.lines import Line2D
from matplotlib.patches import Patch

# Common bar settings
bar_width = 0.15
bar_spacing = 0.3

# Group colors
group_colors = {
    'IBE-1': '#5275a0',
    'IBE-2': '#537d55',
    'IBE-3': '#56a484',
    'IBE-4': '#cb6f5f',
    'TBE-1': '#6baed6',
    'TBE-2': '#74c476',
    'TBE-3': '#fd8d3c'
}

# LIAR-RAW dataset
baseline_liar_raw = 0.42
f1_scores_liar_raw = {
    'IBE-1': {'Mistral': 0.22},
    'IBE-2': {'Llama': 0.22},
    'IBE-3': {'Mistral, Llama, Qwen, Falcon': 0.21},
    'IBE-4': {'Mistral and Falcon': 0.14},
    'TBE-1': {'Llama$_{LoRA}$': 0.30},
    'TBE-2': {'Mistral$_{LoRA}$': 0.32},
    'TBE-3': {
        'RoBERTa-L$_{Mistral}$': 0.47,
        'RoBERTa-L$_{Llama}$': 0.52,
        'RoBERTa-L$_{Gemma}$': 0.48,
        'RoBERTa-L$_{Qwen}$': 0.46,
        'RoBERTa-L$_{Falcon}$': 0.44,
        'XLNet-L$_{Mistral}$': 0.47,
        'XLNet-L$_{Llama}$': 0.54,
        'RoBERTa-L$_{Gemma}$': 0.48,
        'XLNet-L$_{Qwen}$': 0.48,
        'XLNet-L$_{Falcon}$': 0.44,
    }
}

# RAW-FC dataset
baseline_rawfc = 0.61
f1_scores_rawfc = {
    'IBE-1': {'Qwen': 0.59},
    'IBE-2': {'Llama': 0.62},
    'IBE-3': {'Qwen': 0.52},
    'IBE-4': {'Qwen and Mistral': 0.43},
    'TBE-1': {
        'Mistral$_{LoRA}$': 0.65,
        'Llama$_{LoRA}$': 0.65,
        'Qwen$_{LoRA}$': 0.66,
        'Llama$_{LoRA+}$': 0.65,
        'Qwen$_{LoRA+}$': 0.65
    },
    'TBE-2': {
        'XLNet-L$_{Llama}$': 0.62,
        'Llama$_{LoRA+}$': 0.71
    },
    'TBE-3': {
        'RoBERTa-L$_{Mistral}$': 0.83,
        'RoBERTa-L$_{Llama}$': 0.88,
        'RoBERTa-L$_{Qwen}$': 0.71,
        'RoBERTa-L$_{Falcon}$': 0.64,
        'XLNet-L$_{Mistral}$': 0.82,
        'XLNet-L$_{Llama}$': 0.87,
        'XLNet-L$_{Qwen}$': 0.70,
        'XLNet-L$_{Falcon}$': 0.74,
        'Llama$_{LoRA+}$': 0.82  
    }
}

# Plotting function
def plot_f1_scores(dataset_name, f1_data, baseline_value):
    num_bars = sum(len(models) for models in f1_data.values())
    fig_width = max(12, num_bars * 0.55)

    plt.figure(figsize=(fig_width, 8))

    x_labels, heights, colors = [], [], []

    for group, models in f1_data.items():
        for model, score in models.items():
            x_labels.append(model)
            heights.append(score)
            colors.append(group_colors[group])

    x = np.arange(len(x_labels)) * bar_spacing
    plt.bar(x, heights, color=colors, width=bar_width)

    # Draw baseline line
    plt.axhline(y=baseline_value, color='red', linestyle='--', linewidth=2)

    # Add F1 value above bars
    for i, height in enumerate(heights):
        plt.text(x[i], height + 0.01, f"{height:.2f}", ha='center', va='bottom', fontsize=9)

    # X-axis formatting
    plt.xticks(x, x_labels, rotation=45, ha='right')
    plt.ylim(0.05, 0.60)
    plt.ylabel('Macro-F1 Score')
    plt.title(f'Macro-F1 Scores Including IBE and TBE for {dataset_name}')
    plt.grid(axis='y', linestyle='--', alpha=0.6)
    plt.tight_layout()

    # Legend with baseline
    legend_elements = [Patch(facecolor=color, label=group) for group, color in group_colors.items()]
    legend_elements.append(Line2D([0], [0], color='red', linestyle='--', linewidth=2,
                                  label=f'Baseline = {baseline_value:.2f}'))
    plt.legend(handles=legend_elements)
    plt.show()

# Plot LIAR-RAW
plot_f1_scores('LIAR-RAW', f1_scores_liar_raw, baseline_liar_raw)

# Plot RAW-FC
plot_f1_scores('RAW-FC', f1_scores_rawfc, baseline_rawfc)


# Code: Macro-Recall Only

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
from matplotlib.lines import Line2D
from matplotlib.patches import Patch

# Settings
bar_width = 0.15
bar_spacing = 0.3
group_colors = {
    'IBE-1': '#5275a0',
    'IBE-2': '#537d55',
    'IBE-3': '#56a484',
    'IBE-4': '#cb6f5f',
    'TBE-1': '#6baed6',
    'TBE-2': '#74c476',
    'TBE-3': '#fd8d3c'
}
baseline_liar_raw = 0.37
baseline_rawfc = 0.61

# ---- Macro-Recall (MR) ----
mr_liar_raw = {
    'IBE-1': {'mistral': 0.25},
    'IBE-2': {'mistral, llama, qwen': 0.23},
    'IBE-3': {'mistral and falcon': 0.23},
    'IBE-4': {'mistral and falcon': 0.17},
    'TBE-1': {
        'Llama$_{LORA}$': 0.30,
        'Falcon$_{LORA+}$': 0.30
    },
    'TBE-2': {'Mistral$_{LORA}$': 0.32},
    'TBE-3': {
        'RoBERTa-L$_{Mistral}$': 0.47,
        'RoBERTa-L$_{Llama}$': 0.53,
        'RoBERTa-L$_{Gemma}$': 0.50,
        'RoBERTa-L$_{Qwen}$': 0.47,
        'RoBERTa-L$_{Falcon}$': 0.43,
        'XLNet-L$_{Mistral}$': 0.47,
        'XLNet-L$_{Llama}$': 0.54,
        'XLNet-L$_{Qwen}$': 0.48,
        'XLNet-L$_{Gemma}$': 0.43,
        'XLNet-L$_{Falcon}$': 0.44
    }
}

mr_rawfc = {
    'IBE-1': {'Qwen': 0.58},
    'IBE-2': {'Llama': 0.63},
    'IBE-3': {'Qwen': 0.54},
    'IBE-4': {'Mistral and Qwen': 0.46},
    'TBE-1': {
        'Mistral$_{LORA}$': 0.65,
        'Llama$_{LORA}$': 0.64,
        'Qwen$_{LORA}$': 0.66,
        'Llama$_{LORA+}$': 0.64,
        'Qwen$_{LORA+}$': 0.65,
        'Falcon$_{LORA}$': 0.62
    },
    'TBE-2': {
        'XLNet-L$_{Llama}$': 0.63,
        'Llama$_{LORA+}$': 0.71
    },
    'TBE-3': {
        'RoBERTa-L$_{Mistral}$': 0.82,
        'RoBERTa-L$_{Llama}$': 0.88,
        'RoBERTa-L$_{Qwen}$': 0.71,
        'RoBERTa-L$_{Falcon}$': 0.64,
        'XLNet-L$_{Mistral}$': 0.82,
        'XLNet-L$_{Llama}$': 0.88,
        'XLNet-L$_{Qwen}$': 0.70,
        'XLNet-L$_{Falcon}$': 0.75,
        'Llama$_{LORA+}$': 0.82
    }
}

# Plotting Function
def plot_metric_scores(dataset_name, metric_data, baseline_value, metric_label):
    num_bars = sum(len(models) for models in metric_data.values())
    fig_width = max(12, num_bars * 0.55)
    plt.figure(figsize=(fig_width, 8))

    x_labels, heights, colors = [], [], []
    for group, models in metric_data.items():
        for model, score in models.items():
            x_labels.append(model)
            heights.append(score)
            colors.append(group_colors[group])

    x = np.arange(len(x_labels)) * bar_spacing
    plt.bar(x, heights, color=colors, width=bar_width)
    plt.axhline(y=baseline_value, color='red', linestyle='--', linewidth=2)

    for i, height in enumerate(heights):
        plt.text(x[i], height + 0.01, f"{height:.2f}", ha='center', va='bottom', fontsize=9)

    plt.xticks(x, x_labels, rotation=45, ha='right')
    plt.ylim(0, 1)
    plt.ylabel(metric_label)
    plt.title(f'{metric_label} Scores Including IBE and TBE for {dataset_name}')
    plt.grid(axis='y', linestyle='--', alpha=0.6)
    plt.tight_layout()

    legend_elements = [Patch(facecolor=color, label=group) for group, color in group_colors.items()]
    legend_elements.append(Line2D([0], [0], color='red', linestyle='--', linewidth=2,
                                  label=f'Baseline = {baseline_value:.2f}'))
    plt.legend(handles=legend_elements)
    plt.show()

# Generate MR plots
plot_metric_scores('LIAR-RAW', mr_liar_raw, baseline_liar_raw, 'Macro-Recall')
plot_metric_scores('RAW-FC', mr_rawfc, baseline_rawfc, 'Macro-Recall')


# Code for Macro-Precision

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
from matplotlib.lines import Line2D
from matplotlib.patches import Patch

# === Settings ===
bar_width = 0.15
bar_spacing = 0.3
baseline_liar_raw = 0.47
baseline_rawfc = 0.62

# === Group Colors ===
group_colors = {
    'IBE-1': '#5275a0',
    'IBE-2': '#537d55',
    'IBE-3': '#56a484',
    'IBE-4': '#cb6f5f',
    'TBE-1': '#6baed6',
    'TBE-2': '#74c476',
    'TBE-3': '#fd8d3c'
}

# === Macro-Precision Values ===

mp_liar_raw = {
    'IBE-1': {'Qwen': 0.27},
    'IBE-2': {'Llama': 0.30},
    'IBE-3': {'Mistral': 0.40},
    'IBE-4': {'Mistral': 0.30},
    'TBE-1': {'Mistral$_{LORA}$': 0.44},
    'TBE-2': {'Mistral$_{LORA}$': 0.36},
    'TBE-3': {
        'RoBERTa-L$_{Mistral}$': 0.48,
        'RoBERTa-L$_{Llama}$': 0.53,
        'RoBERTa-L$_{Gemma}$': 0.49,
        'RoBERTa-L$_{Qwen}$': 0.48,
        'RoBERTa-L$_{Falcon}$': 0.49,
        'XLNet-L$_{Mistral}$': 0.49,
        'XLNet-L$_{Llama}$': 0.55,
        'XLNet-L$_{Qwen}$': 0.50
    }
}

mp_rawfc = {
    'IBE-1': {'Qwen': 0.61},
    'IBE-2': {'Llama': 0.62},
    'IBE-3': {'Falcon': 0.60},
    'IBE-4': {'Qwen': 0.50},
    'TBE-1': {
        'Mistral$_{LORA}$': 0.69,
        'Llama$_{LORA}$': 0.68,
        'Qwen$_{LORA}$': 0.67,
        'Llama$_{LORA+}$': 0.67,
        'Qwen$_{LORA+}$': 0.70,
        'Falcon$_{LORA+}$': 0.64
    },
    'TBE-2': {
        'XLNet-L$_{Llama}$': 0.63,
        'Llama$_{LORA+}$': 0.73,
        'Mistral$_{LORA+}$': 0.66
    },
    'TBE-3': {
        'RoBERTa-L$_{Mistral}$': 0.83,
        'RoBERTa-L$_{Llama}$': 0.88,
        'RoBERTa-L$_{Qwen}$': 0.73,
        'RoBERTa-L$_{Falcon}$': 0.65,
        'XLNet-L$_{Mistral}$': 0.83,
        'XLNet-L$_{Llama}$': 0.88,
        'XLNet-L$_{Qwen}$': 0.70,
        'XLNet-L$_{Falcon}$': 0.76,
        'Llama$_{LORA+}$': 0.82,
        'Mistral$_{LORA}$': 0.69,
        'Llama$_{LORA}$': 0.63,
        'Llama$_{LORA+}$': 0.84
    }
}

# === Plotting Function ===

def plot_metric_scores(dataset_name, metric_data, baseline_value, metric_label):
    num_bars = sum(len(models) for models in metric_data.values())
    fig_width = max(12, num_bars * 0.55)
    plt.figure(figsize=(fig_width, 8))

    x_labels, heights, colors = [], [], []
    for group, models in metric_data.items():
        for model, score in models.items():
            x_labels.append(model)
            heights.append(score)
            colors.append(group_colors[group])

    x = np.arange(len(x_labels)) * bar_spacing
    plt.bar(x, heights, color=colors, width=bar_width)
    plt.axhline(y=baseline_value, color='red', linestyle='--', linewidth=2)

    for i, height in enumerate(heights):
        plt.text(x[i], height + 0.01, f"{height:.2f}", ha='center', va='bottom', fontsize=9)

    plt.xticks(x, x_labels, rotation=45, ha='right')
    plt.ylim(0, 1)
    plt.ylabel(metric_label)
    plt.title(f'{metric_label} Scores Including IBE and TBE for {dataset_name}')
    plt.grid(axis='y', linestyle='--', alpha=0.6)
    plt.tight_layout()

    legend_elements = [Patch(facecolor=color, label=group) for group, color in group_colors.items()]
    legend_elements.append(Line2D([0], [0], color='red', linestyle='--', linewidth=2,
                                  label=f'Baseline = {baseline_value:.2f}'))
    plt.legend(handles=legend_elements)
    plt.show()

# === Generate MP Plots ===
plot_metric_scores('LIAR-RAW', mp_liar_raw, baseline_liar_raw, 'Macro-Precision')
plot_metric_scores('RAW-FC', mp_rawfc, baseline_rawfc, 'Macro-Precision')


# Below generated graph has been inserted in to the paper

## fresh code

In [ ]:
# LIAR-RAW PLOT (Updated with axis label, larger fonts, clearer ticks)
import matplotlib.pyplot as plt
import numpy as np
from matplotlib.lines import Line2D
from matplotlib.patches import Patch

# LIAR-RAW data
bar_width = 0.15
bar_spacing = 0.3
baseline_liar_raw = 0.42

group_colors = {
    'IBE-1': '#5275a0',
    'IBE-2': '#537d55',
    'IBE-3': '#56a484',
    'IBE-4': '#cb6f5f',
    'TBE-1': '#6baed6',
    'TBE-2': '#74c476',
    'TBE-3': '#fd8d3c'
}

f1_scores_liar_raw = {
    'IBE-1': {'Mistral': 0.22},
    'IBE-2': {'Llama': 0.22},
    'IBE-3': {'Mistral, Llama\nQwen, Falcon': 0.21},
    'IBE-4': {'Mistral and Falcon': 0.14},
    'TBE-1': {'Llama$_{LoRA}$': 0.30},
    'TBE-2': {'Mistral$_{LoRA}$': 0.32},
    'TBE-3': {
        'RoBERTa-L$_{Mistral}$': 0.47,
        'RoBERTa-L$_{Llama}$': 0.52,
        'RoBERTa-L$_{Gemma}$': 0.48,
        'RoBERTa-L$_{Qwen}$': 0.46,
        'RoBERTa-L$_{Falcon}$': 0.44,
        'XLNet-L$_{Mistral}$': 0.47,
        'XLNet-L$_{Llama}$': 0.54,
        'XLNet-L$_{Qwen}$': 0.48,
        'XLNet-L$_{Falcon}$': 0.44
    }
}

# Prepare data
x_labels, heights, colors = [], [], []
for group, models in f1_scores_liar_raw.items():
    for model, score in models.items():
        x_labels.append(model)
        heights.append(score)
        colors.append(group_colors[group])

x = np.arange(len(x_labels)) * bar_spacing

# Plot setup
plt.figure(figsize=(16, 9))
plt.bar(x, heights, color=colors, width=bar_width)
plt.axhline(y=baseline_liar_raw, color='red', linestyle='--', linewidth=2)

# Value labels above bars
for i, height in enumerate(heights):
    plt.text(x[i], height + 0.01, f"{height:.2f}", ha='center', va='bottom', fontsize=14, fontweight='bold')

# Axis formatting
plt.xticks(x, x_labels, rotation=45, ha='right', fontsize=16, fontweight='bold')
plt.yticks(fontsize=16,fontweight='bold')
plt.ylim(0.1, 0.6)
plt.xlabel("Models", fontsize=15, labelpad=10,fontweight='bold')
plt.ylabel("Macro-F1 Score", fontsize=15, labelpad=10,fontweight='bold')
#plt.title("LIAR-RAW: Macro-F1 Scores for IBE and TBE Groups", fontsize=16, pad=20)
plt.grid(axis='y', linestyle='--', alpha=0.6)

# Legend
legend_elements = [Patch(facecolor=color, label=group) for group, color in group_colors.items()]
legend_elements.append(Line2D([0], [0], color='red', linestyle='--', linewidth=2,
                               label=rf"Baseline = $\mathbf{{{baseline_liar_raw:.2f}}}$"))
plt.legend(handles=legend_elements, prop={'size': 16})

plt.tight_layout()


plt.savefig("MF1_lair_raw.pdf", format="pdf", bbox_inches="tight")  # <- Save as PDF
plt.show()

In [ ]:
# RAW-FC PLOT (Updated with label wrapping, font size tuning, axis label)
import matplotlib.pyplot as plt
import numpy as np
from matplotlib.lines import Line2D
from matplotlib.patches import Patch

# RAW-FC data
bar_width = 0.15
bar_spacing = 0.3
baseline_rawfc = 0.61

group_colors = {
    'IBE-1': '#5275a0',
    'IBE-2': '#537d55',
    'IBE-3': '#56a484',
    'IBE-4': '#cb6f5f',
    'TBE-1': '#6baed6',
    'TBE-2': '#74c476',
    'TBE-3': '#fd8d3c'
}

f1_scores_rawfc = {
    'IBE-1': {'Qwen': 0.59},
    'IBE-2': {'Llama': 0.62},
    'IBE-3': {'Qwen': 0.52},
    'IBE-4': {'Qwen\nMistral': 0.43},  # Label split into two lines
    'TBE-1': {
        'Mistral$_{LoRA}$': 0.65,
        'Llama$_{LoRA}$': 0.65,
        'Qwen$_{LoRA}$': 0.66,
        'Llama$_{LoRA+}$': 0.65,
        'Qwen$_{LoRA+}$': 0.65
    },
    'TBE-2': {
        'XLNet-L$_{Llama}$': 0.62,
        'Llama$_{LoRA+}$': 0.71
    },
    'TBE-3': {
        'RoBERTa-L$_{Mistral}$': 0.83,
        'RoBERTa-L$_{Llama}$': 0.88,
        'RoBERTa-L$_{Qwen}$': 0.71,
        'RoBERTa-L$_{Falcon}$': 0.64,
        'XLNet-L$_{Mistral}$': 0.82,
        'XLNet-L$_{Llama}$': 0.87,
        'XLNet-L$_{Qwen}$': 0.70,
        'XLNet-L$_{Falcon}$': 0.74,
        'Llama$_{LoRA+}$': 0.82  
    }
}

# Prepare data
x_labels, heights, colors = [], [], []
for group, models in f1_scores_rawfc.items():
    for model, score in models.items():
        x_labels.append(model)
        heights.append(score)
        colors.append(group_colors[group])

x = np.arange(len(x_labels)) * bar_spacing

# Plot setup
plt.figure(figsize=(18, 9))
plt.bar(x, heights, color=colors, width=bar_width)
plt.axhline(y=baseline_rawfc, color='red', linestyle='--', linewidth=2)

# Value labels above bars
for i, height in enumerate(heights):
    plt.text(x[i], height + 0.01, f"{height:.2f}", ha='center', va='bottom', fontsize=14, fontweight='bold')

# Axis formatting
plt.xticks(x, x_labels, rotation=45, ha='right', fontsize=20, fontweight='bold')
plt.yticks(fontsize=20, fontweight='bold')
plt.ylim(0.4, 0.92)
plt.xlabel("Models", fontsize=15, labelpad=12, fontweight='bold')
plt.ylabel("Macro-F1 Score", fontsize=15, labelpad=12, fontweight='bold')
#plt.title("RAW-FC: Macro-F1 Scores for IBE and TBE Groups", fontsize=16, pad=20)
plt.grid(axis='y', linestyle='--', alpha=0.6)

# Legend
legend_elements = [Patch(facecolor=color, label=group) for group, color in group_colors.items()]
legend_elements.append(Line2D([0], [0], color='red', linestyle='--', linewidth=2,
                               label=rf"Baseline = $\mathbf{{{baseline_rawfc:.2f}}}$"))
plt.legend(handles=legend_elements, fontsize=16)

plt.tight_layout()

plt.savefig("Mf1_rawfc.pdf", format="pdf", bbox_inches="tight")
plt.show()

### precison lair_raw graph

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
from matplotlib.lines import Line2D
from matplotlib.patches import Patch

bar_width = 0.15
bar_spacing = 0.3
baseline_liar_raw = 0.47

group_colors = {
    'IBE-1': '#5275a0', 'IBE-2': '#537d55', 'IBE-3': '#56a484', 'IBE-4': '#cb6f5f',
    'TBE-1': '#6baed6', 'TBE-2': '#74c476', 'TBE-3': '#fd8d3c'
}

mp_liar_raw = {
    'IBE-1': {'Qwen': 0.27},
    'IBE-2': {'Llama': 0.30},
    'IBE-3': {'Mistral': 0.40},
    'IBE-4': {'Mistral': 0.30},
    'TBE-1': {'Mistral$_{LoRA}$': 0.44},
    'TBE-2': {'Mistral$_{LoRA}$': 0.36},
    'TBE-3': {
        'RoBERTa-L$_{Mistral}$': 0.48, 'RoBERTa-L$_{Llama}$': 0.53, 'RoBERTa-L$_{Gemma}$': 0.49,
        'RoBERTa-L$_{Qwen}$': 0.48, 'RoBERTa-L$_{Falcon}$': 0.49, 'XLNet-L$_{Mistral}$': 0.49,
        'XLNet-L$_{Llama}$': 0.55, 'XLNet-L$_{Qwen}$': 0.50
    }
}

# Prepare data
x_labels, heights, colors = [], [], []
for group, models in mp_liar_raw.items():
    for model, score in models.items():
        x_labels.append(model)
        heights.append(score)
        colors.append(group_colors[group])
x = np.arange(len(x_labels)) * bar_spacing

plt.figure(figsize=(18, 9))
plt.bar(x, heights, color=colors, width=bar_width)
plt.axhline(y=baseline_liar_raw, color='red', linestyle='--', linewidth=2)

for i, height in enumerate(heights):
    plt.text(x[i], height + 0.01, f"{height:.2f}", ha='center', va='bottom', fontsize=14, fontweight='bold')

plt.xticks(x, x_labels, rotation=45, ha='right', fontsize=20, fontweight='bold')
plt.yticks(fontsize=20, fontweight='bold')
plt.xlabel("Models", fontsize=15, labelpad=12, fontweight='bold')
plt.ylabel("Macro-F1 Score", fontsize=15, labelpad=12, fontweight='bold')
plt.grid(axis='y', linestyle='--', alpha=0.6)
plt.ylim(0.1, 0.6)

legend_elements = [Patch(facecolor=color, label=group) for group, color in group_colors.items()]
legend_elements.append(Line2D([0], [0], color='red', linestyle='--', linewidth=2, label=rf"Baseline = $\mathbf{{{baseline_liar_raw:.2f}}}$"))
plt.legend(handles=legend_elements, fontsize=15)

plt.tight_layout()
plt.savefig("MP_liar_raw.pdf", format="pdf", bbox_inches="tight")
plt.show()


## precison rawfc_graph

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
from matplotlib.lines import Line2D
from matplotlib.patches import Patch

bar_width = 0.15
bar_spacing = 0.3
baseline_rawfc = 0.62

group_colors = {
    'IBE-1': '#5275a0', 'IBE-2': '#537d55', 'IBE-3': '#56a484', 'IBE-4': '#cb6f5f',
    'TBE-1': '#6baed6', 'TBE-2': '#74c476', 'TBE-3': '#fd8d3c'
}

mp_rawfc = {
    'IBE-1': {'Qwen': 0.61},
    'IBE-2': {'Llama': 0.62},
    'IBE-3': {'Falcon': 0.60},
    'IBE-4': {'Qwen': 0.50},
    'TBE-1': {
        'Mistral$_{LoRA}$': 0.69, 'Llama$_{LoRA}$': 0.68, 'Qwen$_{LoRA}$': 0.67,
        'Llama$_{LoRA+}$': 0.67, 'Qwen$_{LoRA+}$': 0.70, 'Falcon$_{LoRA+}$': 0.64
    },
    'TBE-2': {
        'XLNet-L$_{Llama}$': 0.63, 'Llama$_{LoRA+}$': 0.73, 'Mistral$_{LoRA+}$': 0.66
    },
    'TBE-3': {
        'RoBERTa-L$_{Mistral}$': 0.83, 'RoBERTa-L$_{Llama}$': 0.88, 'RoBERTa-L$_{Qwen}$': 0.73,
        'RoBERTa-L$_{Falcon}$': 0.65, 'XLNet-L$_{Mistral}$': 0.83, 'XLNet-L$_{Llama}$': 0.88,
        'XLNet-L$_{Qwen}$': 0.70, 'XLNet-L$_{Falcon}$': 0.76, 'Llama$_{LoRA+}$': 0.82,
        'Mistral$_{LoRA}$': 0.69, 'Llama$_{LoRA}$': 0.63, 'Llama$_{LoRA+}$': 0.84
    }
}

x_labels, heights, colors = [], [], []
for group, models in mp_rawfc.items():
    for model, score in models.items():
        x_labels.append(model)
        heights.append(score)
        colors.append(group_colors[group])
x = np.arange(len(x_labels)) * bar_spacing

plt.figure(figsize=(22, 9))
plt.bar(x, heights, color=colors, width=bar_width)
plt.axhline(y=baseline_rawfc, color='red', linestyle='--', linewidth=2)

for i, height in enumerate(heights):
    plt.text(x[i], height + 0.01, f"{height:.2f}", ha='center', va='bottom', fontsize=12, fontweight='bold')

plt.xticks(x, x_labels, rotation=45, ha='right', fontsize=20, fontweight='bold')
plt.yticks(fontsize=20, fontweight='bold')
plt.xlabel("Models", fontsize=15, labelpad=12, fontweight='bold')
plt.ylabel("Macro-F1 Score", fontsize=15, labelpad=12, fontweight='bold')
plt.grid(axis='y', linestyle='--', alpha=0.6)
plt.ylim(0.4, 0.92)

legend_elements = [Patch(facecolor=color, label=group) for group, color in group_colors.items()]
legend_elements.append(Line2D([0], [0], color='red', linestyle='--', linewidth=2, label=rf"Baseline = $\mathbf{{{baseline_rawfc:.2f}}}$"))
plt.legend(handles=legend_elements, fontsize=16, loc='upper left', bbox_to_anchor=(0.01, 0.99))

plt.tight_layout()
plt.savefig("MP_rawfc.pdf", format="pdf", bbox_inches="tight")
plt.show()


### recall lair_raw graph

In [ ]:
# Use same imports as above

baseline_liar_raw = 0.37

mr_liar_raw = {
    'IBE-1': {'mistral': 0.25},
    'IBE-2': {'mistral, llama\n qwen': 0.23},
    'IBE-3': {'mistral, falcon': 0.23},
    'IBE-4': {'mistral, falcon': 0.17},
    'TBE-1': {'Llama$_{LoRA}$': 0.30, 'Falcon$_{LoRA+}$': 0.30},
    'TBE-2': {'Mistral$_{LoRA}$': 0.32},
    'TBE-3': {
        'RoBERTa-L$_{Mistral}$': 0.47, 'RoBERTa-L$_{Llama}$': 0.53, 'RoBERTa-L$_{Gemma}$': 0.50,
        'RoBERTa-L$_{Qwen}$': 0.47, 'RoBERTa-L$_{Falcon}$': 0.43, 'XLNet-L$_{Mistral}$': 0.47,
        'XLNet-L$_{Llama}$': 0.54, 'XLNet-L$_{Qwen}$': 0.48, 'XLNet-L$_{Gemma}$': 0.43,
        'XLNet-L$_{Falcon}$': 0.44
    }
}

x_labels, heights, colors = [], [], []
for group, models in mr_liar_raw.items():
    for model, score in models.items():
        x_labels.append(model)
        heights.append(score)
        colors.append(group_colors[group])
x = np.arange(len(x_labels)) * bar_spacing

plt.figure(figsize=(20, 9))
plt.bar(x, heights, color=colors, width=bar_width)
plt.axhline(y=baseline_liar_raw, color='red', linestyle='--', linewidth=2)

for i, height in enumerate(heights):
    plt.text(x[i], height + 0.01, f"{height:.2f}", ha='center', va='bottom', fontsize=12, fontweight='bold')

plt.xticks(x, x_labels, rotation=45, ha='right', fontsize=20, fontweight='bold')
plt.xlabel("Models", fontsize=15, labelpad=12, fontweight='bold')
plt.ylabel("Macro-F1 Score", fontsize=15, labelpad=12, fontweight='bold')
plt.ylabel("Macro-Recall", fontsize=15, fontweight='bold')
plt.grid(axis='y', linestyle='--', alpha=0.6)
plt.ylim(0.1, 0.6)

legend_elements = [Patch(facecolor=color, label=group) for group, color in group_colors.items()]
legend_elements.append(Line2D([0], [0], color='red', linestyle='--', linewidth=2, label=rf"Baseline = $\mathbf{{{baseline_liar_raw:.2f}}}$"))
plt.legend(handles=legend_elements, fontsize=16)

plt.tight_layout()
plt.savefig("MR_liar_raw.pdf", format="pdf", bbox_inches="tight")
plt.show()


### recall rawfc graph

In [ ]:
# Use same imports as above

baseline_rawfc = 0.61

mr_rawfc = {
    'IBE-1': {'Qwen': 0.58},
    'IBE-2': {'Llama': 0.63},
    'IBE-3': {'Qwen': 0.54},
    'IBE-4': {'Mistral Qwen': 0.46},
    'TBE-1': {
        'Mistral$_{LORA}$': 0.65, 'Llama$_{LORA}$': 0.64, 'Qwen$_{LORA}$': 0.66,
        'Llama$_{LORA+}$': 0.64, 'Qwen$_{LORA+}$': 0.65, 'Falcon$_{LORA}$': 0.62
    },
    'TBE-2': {'XLNet-L$_{Llama}$': 0.63, 'Llama$_{LORA+}$': 0.71},
    'TBE-3': {
        'RoBERTa-L$_{Mistral}$': 0.82, 'RoBERTa-L$_{Llama}$': 0.88, 'RoBERTa-L$_{Qwen}$': 0.71,
        'RoBERTa-L$_{Falcon}$': 0.64, 'XLNet-L$_{Mistral}$': 0.82, 'XLNet-L$_{Llama}$': 0.88,
        'XLNet-L$_{Qwen}$': 0.70, 'XLNet-L$_{Falcon}$': 0.75, 'Llama$_{LORA+}$': 0.82
    }
}

x_labels, heights, colors = [], [], []
for group, models in mr_rawfc.items():
    for model, score in models.items():
        x_labels.append(model)
        heights.append(score)
        colors.append(group_colors[group])
x = np.arange(len(x_labels)) * bar_spacing

plt.figure(figsize=(22, 9))
plt.bar(x, heights, color=colors, width=bar_width)
plt.axhline(y=baseline_rawfc, color='red', linestyle='--', linewidth=2)

for i, height in enumerate(heights):
    plt.text(x[i], height + 0.01, f"{height:.2f}", ha='center', va='bottom', fontsize=12, fontweight='bold')

plt.xticks(x, x_labels, rotation=45, ha='right', fontsize=20, fontweight='bold')
plt.yticks(fontsize=20, fontweight='bold')
plt.xlabel("Models", fontsize=15, labelpad=12, fontweight='bold')
plt.ylabel("Macro-F1 Score", fontsize=15, labelpad=12, fontweight='bold')
plt.grid(axis='y', linestyle='--', alpha=0.6)
plt.ylim(0.4, 0.92)

legend_elements = [Patch(facecolor=color, label=group) for group, color in group_colors.items()]
legend_elements.append(Line2D([0], [0], color='red', linestyle='--', linewidth=2, label=rf"Baseline = $\mathbf{{{baseline_rawfc:.2f}}}$"))
plt.legend(handles=legend_elements, fontsize=16)

plt.tight_layout()
plt.savefig("MR_rawfc.pdf", format="pdf", bbox_inches="tight")
plt.show()


## spider graph 

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

# Labels and model scores
categories = ["Informativeness", "Accuracy", "Readability", "Objectivity", "Logicality"]
models_scores = {
    "LLaMA": [4.475, 4.14, 4.43, 4.28, 4.325],
    "Qwen": [3.054, 3.304, 4.161, 3.589, 3.696],
    "Mistral": [3.587, 3.413, 3.935, 4.029, 3.978],
    "Gemma": [3.232, 2.717, 4.051, 3.828, 3.525],
    "Falcon": [3.162, 3.357, 3.662, 3.812, 3.429]
}

# Number of axes
num_vars = len(categories)

# Create angle list and close the polygon
angles = np.linspace(0, 2 * np.pi, num_vars, endpoint=False).tolist()
angles += angles[:1]

# Setup figure
fig, ax = plt.subplots(figsize=(8, 8), subplot_kw=dict(polar=True))

# Configure radial axis (dotted lines and hidden labels)
ax.set_rgrids([1, 2, 3, 4, 5], angle=0)
ax.yaxis.grid(True, linestyle='dotted', color='black')
ax.xaxis.grid(True, linestyle='dotted', color='black')
ax.set_yticklabels([])  # Hide radial labels

# Set angular labels (the pentagon corners)
ax.set_xticks(angles[:-1])
ax.set_xticklabels(categories, fontsize=12)

# Plot each model
for model, scores in models_scores.items():
    values = scores + scores[:1]
    ax.plot(angles, values, label=model)
    ax.fill(angles, values, alpha=0.1)

# Add title and legend
ax.set_title("Evaluation Results Across Five Dimensions (200 Samples)", size=14, pad=20)
ax.legend(loc='upper right', bbox_to_anchor=(1.2, 1.1))

plt.tight_layout()
plt.show()


In [ ]:
import matplotlib.pyplot as plt
import numpy as np

# Define categories and their count
categories = ["Informativeness", "Accuracy", "Readability", "Objectivity", "Logicality"]
N = len(categories)

# Add angle for closing the circle
angles = np.linspace(0, 2 * np.pi, N, endpoint=False).tolist()
angles += angles[:1]

# Model scores
model_scores = {
    "LLaMA": [4.475, 4.14, 4.43, 4.28, 4.325],
    "Qwen": [3.054, 3.304, 4.161, 3.589, 3.696],
    "Mistral": [3.587, 3.413, 3.935, 4.029, 3.978],
    "Gemma": [3.232, 2.717, 4.051, 3.828, 3.525],
    "Falcon": [3.162, 3.357, 3.662, 3.812, 3.429]
}

# Colors for each model
colors = {
    "LLaMA": "#5275a0",
    "Qwen": "#537d55",
    "Mistral": "#56a484",
    "Gemma": "#cb6f5f",
    "Falcon": "#6baed6"
}

# Create figure
fig, ax = plt.subplots(figsize=(8, 8), subplot_kw=dict(polar=True))

# Draw dotted grid and hide radial labels
ax.set_yticklabels([])
ax.yaxis.grid(True, linestyle='dotted', color='black')
ax.xaxis.grid(True, linestyle='dotted', color='black')

# Set labels on the axes
ax.set_xticks(angles[:-1])
ax.set_xticklabels(categories, fontsize=12)

# Plot each model
for model, scores in model_scores.items():
    values = scores + scores[:1]  # Close the loop
    ax.plot(angles, values, label=model, color=colors[model], linewidth=1.5)
    ax.fill(angles, values, color=colors[model], alpha=0.1)

# Add title
ax.set_title("Evaluation Results of Models Across Five Dimensions", size=15, pad=30, fontweight='bold')

# Legend aligned to the left
plt.legend(loc='center left', bbox_to_anchor=(-0.3, 0.7), fontsize=11)

# Show plot
plt.tight_layout()
plt.show()


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from math import pi

# Data for the models
labels = ['Informativeness', 'Accuracy', 'Readability', 'Objectivity', 'Logicality']
num_vars = len(labels)

# Average scores for each model
llama = [4.475, 4.14, 4.43, 4.28, 4.325]
qwen = [3.054, 3.304, 4.161, 3.589, 3.696]
mistral = [3.587, 3.413, 3.935, 4.029, 3.978]
gemma = [3.232, 2.717, 4.051, 3.828, 3.525]
falcon = [3.162, 3.357, 3.662, 3.812, 3.429]

# Combine data into a list
data = [llama, qwen, mistral, gemma, falcon]
model_names = ['LLaMA', 'Qwen', 'Mistral', 'Gemma', 'Falcon']
colors = ['#FF9999', '#66B2FF', '#99FF99', '#FFCC99', '#FF99CC']  # Colors for each model

# Compute angle for each category
angles = [n / float(num_vars) * 2 * pi for n in range(num_vars)]
angles += angles[:1]  # Complete the loop

# Initialize the radar chart
fig, ax = plt.subplots(figsize=(6, 6), subplot_kw=dict(polar=True))

# Draw one axe per variable and add labels
ax.set_theta_offset(pi / 2)
ax.set_theta_direction(-1)

# Draw axis lines and labels
plt.xticks(angles[:-1], labels)

# Draw ylabels (scale)
ax.set_rlabel_position(0)
plt.yticks([1, 2, 3, 4, 5], ["1", "2", "3", "4", "5"], color="grey", size=7)
plt.ylim(0, 5)

# Plot each model's data
for i, model_data in enumerate(data):
    values = model_data + model_data[:1]  # Complete the loop
    ax.plot(angles, values, linewidth=2, linestyle='solid', label=model_names[i], color=colors[i])
    ax.fill(angles, values, color=colors[i], alpha=0.25)

# Add a legend
plt.legend(loc='upper right', bbox_to_anchor=(0.1, 0.1))

# Show the plot
plt.show()

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from math import pi

# Data for the models
labels = ['Informativeness', 'Accuracy', 'Readability', 'Objectivity', 'Logicality']
num_vars = len(labels)

# Average scores for each model
llama = [4.475, 4.14, 4.43, 4.28, 4.325]
qwen = [3.054, 3.304, 4.161, 3.589, 3.696]
mistral = [3.587, 3.413, 3.935, 4.029, 3.978]
gemma = [3.232, 2.717, 4.051, 3.828, 3.525]
falcon = [3.162, 3.357, 3.662, 3.812, 3.429]

# Combine data into a list
data = [llama, qwen, mistral, gemma, falcon]
model_names = ['LLaMA', 'Qwen', 'Mistral', 'Gemma', 'Falcon']
colors = ['#FF9999', '#66B2FF', '#99FF99', '#FFCC99', '#FF99CC']  # TBE (red), IBE (blue), then others

# Compute angle for each category
angles = [n / float(num_vars) * 2 * pi for n in range(num_vars)]
angles += angles[:1]  # Complete the loop

# Initialize the radar chart
fig, ax = plt.subplots(figsize=(6, 6), subplot_kw=dict(polar=True))

# Draw one axe per variable and add labels
ax.set_theta_offset(pi / 2)
ax.set_theta_direction(-1)

# Draw axis labels
plt.xticks(angles[:-1], labels)

# Remove default circular grid
ax.grid(False)

# Draw pentagon boundaries manually
for r in range(1, 6):
    pentagon = [r] * num_vars  # Constant radius for each level
    pentagon = np.append(pentagon, pentagon[0])  # Complete the loop
    ax.plot(angles, pentagon, color='gray', linestyle='--', linewidth=1)

# Draw bold dashed spokes (radial lines)
for angle in angles[:-1]:
    ax.plot([angle, angle], [0, 5], color='gray', linestyle='--', linewidth=2)

# Draw ylabels (scale)
ax.set_rlabel_position(0)
plt.yticks([1, 2, 3, 4, 5], ["1", "2", "3", "4", "5"], color="grey", size=7)
plt.ylim(0, 5)

# Plot each model's data
for i, model_data in enumerate(data):
    values = model_data + model_data[:1]  # Complete the loop
    # Plot the line and fill
    ax.plot(angles, values, linewidth=2, linestyle='solid', label=model_names[i], color=colors[i])
    ax.fill(angles, values, color=colors[i], alpha=0.25)
    # Add bold points where the line touches the axes
    for j in range(num_vars):
        ax.plot(angles[j], model_data[j], marker='o', markersize=8, color=colors[i])

# Add a legend
plt.legend(loc='upper right', bbox_to_anchor=(0.1, 0.1))

# Save the plot
plt.savefig('radar_chart_pentagon.png')

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from math import pi

# Data for the models
labels = ['Informativeness', 'Accuracy', 'Readability', 'Objectivity', 'Logicality']
num_vars = len(labels)

# Average scores for each model
llama = [4.475, 4.14, 4.43, 4.28, 4.325]
qwen = [3.054, 3.304, 4.161, 3.589, 3.696]
mistral = [3.587, 3.413, 3.935, 4.029, 3.978]
gemma = [3.232, 2.717, 4.051, 3.828, 3.525]
falcon = [3.162, 3.357, 3.662, 3.812, 3.429]

# Combine data into a list
data = [llama, qwen, mistral, gemma, falcon]
model_names = ['LLaMA', 'Qwen', 'Mistral', 'Gemma', 'Falcon']
colors = ['#FF9999', '#66B2FF', '#99FF99', '#FFCC99', '#FF99CC']  # TBE (red), IBE (blue), then others

# Compute angle for each category
angles = [n / float(num_vars) * 2 * pi for n in range(num_vars)]
angles += angles[:1]  # Complete the loop

# Initialize the radar chart
fig, ax = plt.subplots(figsize=(6, 6), subplot_kw=dict(polar=True))

# Draw one axe per variable and add labels
ax.set_theta_offset(pi / 2)
ax.set_theta_direction(-1)

# Draw axis labels
plt.xticks(angles[:-1], labels)

# Remove default circular grid
ax.grid(False)

# Draw pentagon boundaries manually
for r in range(1, 6):
    pentagon = [r] * num_vars  # Constant radius for each level
    pentagon = np.append(pentagon, pentagon[0])  # Complete the loop
    ax.plot(angles, pentagon, color='gray', linestyle='--', linewidth=1)

# Draw bold dashed spokes (radial lines)
for angle in angles[:-1]:
    ax.plot([angle, angle], [0, 5], color='gray', linestyle='--', linewidth=2)

# Draw ylabels (scale)
ax.set_rlabel_position(0)
plt.yticks([1, 2, 3, 4, 5], ["1", "2", "3", "4", "5"], color="grey", size=7)
plt.ylim(0, 5)

# Plot each model's data
for i, model_data in enumerate(data):
    values = model_data + model_data[:1]  # Complete the loop
    # Plot the line and fill
    ax.plot(angles, values, linewidth=2, linestyle='solid', label=model_names[i], color=colors[i])
    ax.fill(angles, values, color=colors[i], alpha=0.25)
    # Add bold points where the line touches the axes
    for j in range(num_vars):
        ax.plot(angles[j], model_data[j], marker='o', markersize=8, color=colors[i])

# Add a legend
plt.legend(loc='upper right', bbox_to_anchor=(0.1, 0.1))

# Remove the outer circle (spine)
ax.spines['polar'].set_visible(False)

# Save the plot
plt.savefig('radar_chart_no_outer_circle.png')

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from math import pi

# Data for the models
labels = ['Informativeness', 'Accuracy', 'Readability', 'Objectivity', 'Logicality']
num_vars = len(labels)

# Average scores for each model
llama = [4.475, 4.14, 4.43, 4.28, 4.325]
qwen = [3.054, 3.304, 4.161, 3.589, 3.696]
mistral = [3.587, 3.413, 3.935, 4.029, 3.978]
gemma = [3.232, 2.717, 4.051, 3.828, 3.525]
falcon = [3.162, 3.357, 3.662, 3.812, 3.429]

# Combine data into a list
data = [llama, qwen, mistral, gemma, falcon]
model_names = ['LLaMA', 'Qwen', 'Mistral', 'Gemma', 'Falcon']
colors = ['#FF9999', '#66B2FF', '#99FF99', '#FFCC99', '#FF99CC']  # TBE (red), IBE (blue), then others

# Compute angle for each category
angles = [n / float(num_vars) * 2 * pi for n in range(num_vars)]
angles += angles[:1]  # Complete the loop

# Initialize the radar chart
fig, ax = plt.subplots(figsize=(6, 6), subplot_kw=dict(polar=True))

# Draw one axe per variable and add labels
ax.set_theta_offset(pi / 2)
ax.set_theta_direction(-1)

# Draw axis labels
plt.xticks(angles[:-1], labels,color="black", size=10, weight='bold')

# Remove default circular grid
ax.grid(False)

# Draw pentagon boundaries manually
for r in range(1, 6):
    pentagon = [r] * num_vars  # Constant radius for each level
    pentagon = np.append(pentagon, pentagon[0])  # Complete the loop
    ax.plot(angles, pentagon, color='gray', linestyle='--', linewidth=2)

# Draw bold dashed spokes (radial lines)
for angle in angles[:-1]:
    ax.plot([angle, angle], [0, 5], color='gray', linestyle='--', linewidth=1)

# Draw ylabels (scale) with bold and larger font
ax.set_rlabel_position(0)
plt.yticks([1, 2, 3, 4, 5], ["1", "2", "3", "4", "5"], color="black", size=10, weight='bold')
plt.ylim(0, 5)

# Plot each model's data
for i, model_data in enumerate(data):
    values = model_data + model_data[:1]  # Complete the loop
    # Plot the line and fill
    ax.plot(angles, values, linewidth=2, linestyle='solid', label=model_names[i], color=colors[i])
    ax.fill(angles, values, color=colors[i], alpha=0.25)
    # Add bold points where the line touches the axes
    for j in range(num_vars):
        ax.plot(angles[j], model_data[j], marker='o', markersize=10, color=colors[i])

# Add a legend
plt.legend(loc='upper right', bbox_to_anchor=(0.1, 0.1))

# Remove the outer circle (spine)
ax.spines['polar'].set_visible(False)

# Save the plot
plt.savefig('radar_chart_bold_values.png')

## gemma _rawfc

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from math import pi

# Data for the models
labels = ['Informativeness', 'Accuracy', 'Readability', 'Objectivity', 'Logicality']
num_vars = len(labels)

# Average scores for each model (Gemma data)
gemma_gemma = [2.876, 2.496, 3.597, 3.434, 2.837]
gemma_falcon = [2.142, 2.081, 2.381, 2.254, 1.893]
gemma_qwen = [2.13, 2.043, 2.876, 2.589, 2.486]
gemma_mistral = [2.99, 3.021, 3.123, 3.328, 3.554]
gemma_llama = [2.199, 1.482, 1.89, 1.518, 1.707]

# Combine data into a list
data = [gemma_gemma, gemma_falcon, gemma_qwen, gemma_mistral, gemma_llama]
model_names = ['Gemma', 'Falcon', 'Qwen', 'Mistral', 'LLaMA']
colors = ['#FF9999', '#66B2FF', '#99FF99', '#FFCC99', '#FF99CC']  # TBE (red), IBE (blue), then others

# Compute angle for each category
angles = [n / float(num_vars) * 2 * pi for n in range(num_vars)]
angles += angles[:1]  # Complete the loop

# Initialize the radar chart
fig, ax = plt.subplots(figsize=(6, 6), subplot_kw=dict(polar=True))

# Draw one axe per variable and add labels
ax.set_theta_offset(pi / 2)
ax.set_theta_direction(-1)

# Draw axis labels
plt.xticks(angles[:-1], labels,color="black", size=10, weight='bold')

# Remove default circular grid
ax.grid(False)

# Draw pentagon boundaries manually
for r in range(1, 6):
    pentagon = [r] * num_vars  # Constant radius for each level
    pentagon = np.append(pentagon, pentagon[0])  # Complete the loop
    ax.plot(angles, pentagon, color='gray', linestyle='--', linewidth=2)

# Draw bold dashed spokes (radial lines)
for angle in angles[:-1]:
    ax.plot([angle, angle], [0, 5], color='gray', linestyle='--', linewidth=1)

# Draw ylabels (scale) with bold and larger font
ax.set_rlabel_position(0)
plt.yticks([1, 2, 3, 4, 5], ["1", "2", "3", "4", "5"], color="black", size=10, weight='bold')
plt.ylim(0, 5)

# Plot each model's data
for i, model_data in enumerate(data):
    values = model_data + model_data[:1]  # Complete the loop
    # Plot the line and fill
    ax.plot(angles, values, linewidth=2, linestyle='solid', label=model_names[i], color=colors[i])
    ax.fill(angles, values, color=colors[i], alpha=0.25)
    # Add bold points where the line touches the axes
    for j in range(num_vars):
        ax.plot(angles[j], model_data[j], marker='o', markersize=10, color=colors[i])

# Add a legend
plt.legend(loc='upper right', bbox_to_anchor=(0.1, 0.1))

# Remove the outer circle (spine)
ax.spines['polar'].set_visible(False)

# Save the plot
plt.savefig('radar_chart_gemma_data.png')
plt.savefig('radar_chart_gemma_data_rawfc.pdf', format='pdf', bbox_inches='tight')

## falcon rawfc

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from math import pi

# Data for the models
labels = ['Informativeness', 'Accuracy', 'Readability', 'Objectivity', 'Logicality']
num_vars = len(labels)

# Average scores for each model (Falcon data)
falcon_gemma = [3.0, 2.563, 3.971, 3.796, 3.146]
falcon_falcon = [2.097, 2.426, 3.385, 3.062, 2.446]
falcon_qwen = [2.045, 1.91, 3.445, 2.574, 2.594]
falcon_mistral = [3.045, 3.076, 3.606, 3.732, 3.596]
falcon_llama = [3.735, 2.65, 3.6, 3.0, 3.02]

# Combine data into a list
data = [falcon_gemma, falcon_falcon, falcon_qwen, falcon_mistral, falcon_llama]
model_names = ['Gemma', 'Falcon', 'Qwen', 'Mistral', 'LLaMA']
colors = ['#FF9999', '#66B2FF', '#99FF99', '#FFCC99', '#FF99CC']  # TBE (red), IBE (blue), then others

# Compute angle for each category
angles = [n / float(num_vars) * 2 * pi for n in range(num_vars)]
angles += angles[:1]  # Complete the loop

# Initialize the radar chart
fig, ax = plt.subplots(figsize=(6, 6), subplot_kw=dict(polar=True))

# Draw one axe per variable and add labels
ax.set_theta_offset(pi / 2)
ax.set_theta_direction(-1)

# Draw axis labels
plt.xticks(angles[:-1], labels,color="black", size=10, weight='bold')

# Remove default circular grid
ax.grid(False)

# Draw pentagon boundaries manually
for r in range(1, 6):
    pentagon = [r] * num_vars  # Constant radius for each level
    pentagon = np.append(pentagon, pentagon[0])  # Complete the loop
    ax.plot(angles, pentagon, color='gray', linestyle='--', linewidth=2)

# Draw bold dashed spokes (radial lines)
for angle in angles[:-1]:
    ax.plot([angle, angle], [0, 5], color='gray', linestyle='--', linewidth=1)

# Draw ylabels (scale) with bold and larger font
ax.set_rlabel_position(0)
plt.yticks([1, 2, 3, 4, 5], ["1", "2", "3", "4", "5"], color="black", size=10, weight='bold')
plt.ylim(0, 5)

# Plot each model's data
for i, model_data in enumerate(data):
    values = model_data + model_data[:1]  # Complete the loop
    # Plot the line and fill
    ax.plot(angles, values, linewidth=2, linestyle='solid', label=model_names[i], color=colors[i])
    ax.fill(angles, values, color=colors[i], alpha=0.25)
    # Add bold points where the line touches the axes
    for j in range(num_vars):
        ax.plot(angles[j], model_data[j], marker='o', markersize=10, color=colors[i])

# Add a legend
plt.legend(loc='upper right', bbox_to_anchor=(0.1, 0.1))

# Remove the outer circle (spine)
ax.spines['polar'].set_visible(False)

# Save the plot
plt.savefig('radar_chart_falcon_data.png')
plt.savefig('radar_chart_falcon_data_rawfc.pdf', format='pdf', bbox_inches='tight')

## qwen rawfc

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from math import pi

# Data for the models
labels = ['Informativeness', 'Accuracy', 'Readability', 'Objectivity', 'Logicality']
num_vars = len(labels)

# Average scores for each model (Qwen data)
qwen_gemma = [3.723, 2.892, 4.408, 3.785, 3.508]
qwen_falcon = [3.722, 3.611, 3.616, 4.045, 3.707]
qwen_qwen = [3.303, 3.303, 4.072, 3.569, 3.764]
qwen_mistral = [4.237, 3.636, 4.111, 3.939, 4.146]
qwen_llama = [4.924, 4.101, 4.818, 4.318, 4.52]

# Combine data into a list
data = [qwen_gemma, qwen_falcon, qwen_qwen, qwen_mistral, qwen_llama]
model_names = ['Gemma', 'Falcon', 'Qwen', 'Mistral', 'LLaMA']
colors = ['#FF9999', '#66B2FF', '#99FF99', '#FFCC99', '#FF99CC']  # TBE (red), IBE (blue), then others

# Compute angle for each category
angles = [n / float(num_vars) * 2 * pi for n in range(num_vars)]
angles += angles[:1]  # Complete the loop

# Initialize the radar chart
fig, ax = plt.subplots(figsize=(6, 6), subplot_kw=dict(polar=True))

# Draw one axe per variable and add labels
ax.set_theta_offset(pi / 2)
ax.set_theta_direction(-1)

# Draw axis labels
plt.xticks(angles[:-1], labels,color="black", size=10, weight='bold')

# Remove default circular grid
ax.grid(False)

# Draw pentagon boundaries manually
for r in range(1, 6):
    pentagon = [r] * num_vars  # Constant radius for each level
    pentagon = np.append(pentagon, pentagon[0])  # Complete the loop
    ax.plot(angles, pentagon, color='gray', linestyle='--', linewidth=2)

# Draw bold dashed spokes (radial lines)
for angle in angles[:-1]:
    ax.plot([angle, angle], [0, 5], color='gray', linestyle='--', linewidth=1)

# Draw ylabels (scale) with bold and larger font
ax.set_rlabel_position(0)
plt.yticks([1, 2, 3, 4, 5], ["1", "2", "3", "4", "5"], color="black", size=10, weight='bold')
plt.ylim(0, 5)

# Plot each model's data
for i, model_data in enumerate(data):
    values = model_data + model_data[:1]  # Complete the loop
    # Plot the line and fill
    ax.plot(angles, values, linewidth=2, linestyle='solid', label=model_names[i], color=colors[i])
    ax.fill(angles, values, color=colors[i], alpha=0.25)
    # Add bold points where the line touches the axes
    for j in range(num_vars):
        ax.plot(angles[j], model_data[j], marker='o', markersize=10, color=colors[i])

# Add a legend
plt.legend(loc='upper right', bbox_to_anchor=(0.1, 0.1))

# Remove the outer circle (spine)
ax.spines['polar'].set_visible(False)

# Save the plot
plt.savefig('radar_chart_qwen_data.png')
plt.savefig('radar_chart_qwen_data_rawfc.pdf', format='pdf', bbox_inches='tight')

### mistral

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from math import pi

# Data for the models
labels = ['Informativeness', 'Accuracy', 'Readability', 'Objectivity', 'Logicality']
num_vars = len(labels)

# Average scores for each model (Mistral data)
mistral_gemma = [3.396, 2.739, 4.231, 3.769, 3.291]
mistral_falcon = [3.152, 3.213, 3.655, 3.787, 3.264]
mistral_qwen = [2.932, 2.849, 3.88, 3.323, 3.344]
mistral_mistral = [3.601, 3.106, 3.909, 3.909, 3.808]
mistral_llama = [4.749, 4.121, 4.759, 4.467, 4.442]

# Combine data into a list
data = [mistral_gemma, mistral_falcon, mistral_qwen, mistral_mistral, mistral_llama]
model_names = ['Gemma', 'Falcon', 'Qwen', 'Mistral', 'LLaMA']
colors = ['#FF9999', '#66B2FF', '#99FF99', '#FFCC99', '#FF99CC']  # TBE (red), IBE (blue), then others

# Compute angle for each category
angles = [n / float(num_vars) * 2 * pi for n in range(num_vars)]
angles += angles[:1]  # Complete the loop

# Initialize the radar chart
fig, ax = plt.subplots(figsize=(6, 6), subplot_kw=dict(polar=True))

# Draw one axe per variable and add labels
ax.set_theta_offset(pi / 2)
ax.set_theta_direction(-1)

# Draw axis labels
plt.xticks(angles[:-1], labels,color="black", size=10, weight='bold')

# Remove default circular grid
ax.grid(False)

# Draw pentagon boundaries manually
for r in range(1, 6):
    pentagon = [r] * num_vars  # Constant radius for each level
    pentagon = np.append(pentagon, pentagon[0])  # Complete the loop
    ax.plot(angles, pentagon, color='gray', linestyle='--', linewidth=2)

# Draw bold dashed spokes (radial lines)
for angle in angles[:-1]:
    ax.plot([angle, angle], [0, 5], color='gray', linestyle='--', linewidth=1)

# Draw ylabels (scale) with bold and larger font
ax.set_rlabel_position(0)
plt.yticks([1, 2, 3, 4, 5], ["1", "2", "3", "4", "5"], color="black", size=10, weight='bold')
plt.ylim(0, 5)

# Plot each model's data
for i, model_data in enumerate(data):
    values = model_data + model_data[:1]  # Complete the loop
    # Plot the line and fill
    ax.plot(angles, values, linewidth=2, linestyle='solid', label=model_names[i], color=colors[i])
    ax.fill(angles, values, color=colors[i], alpha=0.25)
    # Add bold points where the line touches the axes
    for j in range(num_vars):
        ax.plot(angles[j], model_data[j], marker='o', markersize=10, color=colors[i])

# Add a legend
plt.legend(loc='upper right', bbox_to_anchor=(0.1, 0.1))

# Remove the outer circle (spine)
ax.spines['polar'].set_visible(False)

# Save the plot
plt.savefig('radar_chart_mistral_data.png')
plt.savefig('radar_chart_mistral_data_rawfc.pdf', format='pdf', bbox_inches='tight')

### llama rawfc 

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from math import pi

# Data for the models
labels = ['Informativeness', 'Accuracy', 'Readability', 'Objectivity', 'Logicality']
num_vars = len(labels)

# Average scores for each model (LLaMA data)
llama_gemma = [3.346, 2.846, 4.115, 3.808, 3.554]
llama_falcon = [3.192, 3.364, 3.611, 3.848, 3.465]
llama_qwen = [3.139, 3.361, 4.153, 3.611, 3.764]
llama_mistral = [3.571, 3.495, 3.955, 4.061, 4.03]
llama_llama = [4.475, 4.14, 4.43, 4.28, 4.325]

# Combine data into a list
data = [llama_gemma, llama_falcon, llama_qwen, llama_mistral, llama_llama]
model_names = ['Gemma', 'Falcon', 'Qwen', 'Mistral', 'LLaMA']
colors = ['#FF9999', '#66B2FF', '#99FF99', '#radar_chart_llama_data_rawfc.pdfFFCC99', '#FF99CC']  # TBE (red), IBE (blue), then others

# Compute angle for each category
angles = [n / float(num_vars) * 2 * pi for n in range(num_vars)]
angles += angles[:1]  # Complete the loop

# Initialize the radar chart
fig, ax = plt.subplots(figsize=(6, 6), subplot_kw=dict(polar=True))

# Draw one axe per variable and add labels
ax.set_theta_offset(pi / 2)
ax.set_theta_direction(-1)

# Draw axis labels
plt.xticks(angles[:-1], labels,color="black", size=10, weight='bold')

# Remove default circular grid
ax.grid(False)

# Draw pentagon boundaries manually
for r in range(1, 6):
    pentagon = [r] * num_vars  # Constant radius for each level
    pentagon = np.append(pentagon, pentagon[0])  # Complete the loop
    ax.plot(angles, pentagon, color='gray', linestyle='--', linewidth=2)

# Draw bold dashed spokes (radial lines)
for angle in angles[:-1]:
    ax.plot([angle, angle], [0, 5], color='gray', linestyle='--', linewidth=1)

# Draw ylabels (scale) with bold and larger font
ax.set_rlabel_position(0)
plt.yticks([1, 2, 3, 4, 5], ["1", "2", "3", "4", "5"], color="black", size=10, weight='bold')
plt.ylim(0, 5)

# Plot each model's data
for i, model_data in enumerate(data):
    values = model_data + model_data[:1]  # Complete the loop
    # Plot the line and fill
    ax.plot(angles, values, linewidth=2, linestyle='solid', label=model_names[i], color=colors[i])
    ax.fill(angles, values, color=colors[i], alpha=0.25)
    # Add bold points where the line touches the axes
    for j in range(num_vars):
        ax.plot(angles[j], model_data[j], marker='o', markersize=10, color=colors[i])

# Add a legend
plt.legend(loc='upper right', bbox_to_anchor=(0.1, 0.1))

# Remove the outer circle (spine)
ax.spines['polar'].set_visible(False)

# Save the plot
plt.savefig('radar_chart_llama_data.png')
plt.savefig('radar_chart_llama_data_rawfc.pdf', format='pdf', bbox_inches='tight')

## AVERAGE

Graph 6: Average Across All Graphs
Now, we’ll calculate the average values for each model across all five graphs and create a sixth radar chart.

Step 1: Collect All Data\n
Let’s list all the data for each model across the five graphs:

Gemma:\n
Graph 1 (gemma_gemma): [2.876, 2.496, 3.597, 3.434, 2.837]\n
Graph 2 (falcon_gemma): [3.0, 2.563, 3.971, 3.796, 3.146]\n
Graph 3 (qwen_gemma): [3.723, 2.892, 4.408, 3.785, 3.508]\n
Graph 4 (mistral_gemma): [3.396, 2.739, 4.231, 3.769, 3.291]\n
Graph 5 (llama_gemma): [3.346, 2.846, 4.115, 3.808, 3.554]\n
Falcon:
Graph 1 (gemma_falcon): [2.142, 2.081, 2.381, 2.254, 1.893]
Graph 2 (falcon_falcon): [2.097, 2.426, 3.385, 3.062, 2.446]
Graph 3 (qwen_falcon): [3.722, 3.611, 3.616, 4.045, 3.707]
Graph 4 (mistral_falcon): [3.152, 3.213, 3.655, 3.787, 3.264]
Graph 5 (llama_falcon): [3.192, 3.364, 3.611, 3.848, 3.465]
Qwen:
Graph 1 (gemma_qwen): [2.13, 2.043, 2.876, 2.589, 2.486]
Graph 2 (falcon_qwen): [2.045, 1.91, 3.445, 2.574, 2.594]
Graph 3 (qwen_qwen): [3.303, 3.303, 4.072, 3.569, 3.764]
Graph 4 (mistral_qwen): [2.932, 2.849, 3.88, 3.323, 3.344]
Graph 5 (llama_qwen): [3.139, 3.361, 4.153, 3.611, 3.764]
Mistral:
Graph 1 (gemma_mistral): [2.99, 3.021, 3.123, 3.328, 3.554]
Graph 2 (falcon_mistral): [3.045, 3.076, 3.606, 3.732, 3.596]
Graph 3 (qwen_mistral): [4.237, 3.636, 4.111, 3.939, 4.146]
Graph 4 (mistral_mistral): [3.601, 3.106, 3.909, 3.909, 3.808]
Graph 5 (llama_mistral): [3.571, 3.495, 3.955, 4.061, 4.03]
LLaMA:
Graph 1 (gemma_llama): [2.199, 1.482, 1.89, 1.518, 1.707]
Graph 2 (falcon_llama): [3.735, 2.65, 3.6, 3.0, 3.02]
Graph 3 (qwen_llama): [4.924, 4.101, 4.818, 4.318, 4.52]
Graph 4 (mistral_llama): [4.749, 4.121, 4.759, 4.467, 4.442]
Graph 5 (llama_llama): [4.475, 4.14, 4.43, 4.28, 4.325]
Step 2: Calculate Averages
We’ll calculate the average for each metric (Informativeness, Accuracy, Readability, Objectivity, Logicality) across the five graphs for each model.

Gemma:
Informativeness: (2.876 + 3.0 + 3.723 + 3.396 + 3.346) / 5 = 3.2682
Accuracy: (2.496 + 2.563 + 2.892 + 2.739 + 2.846) / 5 = 2.7072
Readability: (3.597 + 3.971 + 4.408 + 4.231 + 4.115) / 5 = 4.0644
Objectivity: (3.434 + 3.796 + 3.785 + 3.769 + 3.808) / 5 = 3.7184
Logicality: (2.837 + 3.146 + 3.508 + 3.291 + 3.554) / 5 = 3.2672
Falcon:
Informativeness: (2.142 + 2.097 + 3.722 + 3.152 + 3.192) / 5 = 2.861
Accuracy: (2.081 + 2.426 + 3.611 + 3.213 + 3.364) / 5 = 2.939
Readability: (2.381 + 3.385 + 3.616 + 3.655 + 3.611) / 5 = 3.3296
Objectivity: (2.254 + 3.062 + 4.045 + 3.787 + 3.848) / 5 = 3.3992
Logicality: (1.893 + 2.446 + 3.707 + 3.264 + 3.465) / 5 = 2.955
Qwen:
Informativeness: (2.13 + 2.045 + 3.303 + 2.932 + 3.139) / 5 = 2.7098
Accuracy: (2.043 + 1.91 + 3.303 + 2.849 + 3.361) / 5 = 2.6932
Readability: (2.876 + 3.445 + 4.072 + 3.88 + 4.153) / 5 = 3.6852
Objectivity: (2.589 + 2.574 + 3.569 + 3.323 + 3.611) / 5 = 3.1332
Logicality: (2.486 + 2.594 + 3.764 + 3.344 + 3.764) / 5 = 3.1904
Mistral:
Informativeness: (2.99 + 3.045 + 4.237 + 3.601 + 3.571) / 5 = 3.4888
Accuracy: (3.021 + 3.076 + 3.636 + 3.106 + 3.495) / 5 = 3.2668
Readability: (3.123 + 3.606 + 4.111 + 3.909 + 3.955) / 5 = 3.7408
Objectivity: (3.328 + 3.732 + 3.939 + 3.909 + 4.061) / 5 = 3.7938
Logicality: (3.554 + 3.596 + 4.146 + 3.808 + 4.03) / 5 = 3.8268
LLaMA:
Informativeness: (2.199 + 3.735 + 4.924 + 4.749 + 4.475) / 5 = 4.0164
Accuracy: (1.482 + 2.65 + 4.101 + 4.121 + 4.14) / 5 = 3.2988
Readability: (1.89 + 3.6 + 4.818 + 4.759 + 4.43) / 5 = 3.8994
Objectivity: (1.518 + 3.0 + 4.318 + 4.467 + 4.28) / 5 = 3.5166
Logicality: (1.707 + 3.02 + 4.52 + 4.442 + 4.325) / 5 = 3.6028

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from math import pi

# Data for the models
labels = ['Informativeness', 'Accuracy', 'Readability', 'Objectivity', 'Logicality']
num_vars = len(labels)

# Average scores for each model (averaged across all graphs)
avg_gemma = [3.2682, 2.7072, 4.0644, 3.7184, 3.2672]
avg_falcon = [2.861, 2.939, 3.3296, 3.3992, 2.955]
avg_qwen = [2.7098, 2.6932, 3.6852, 3.1332, 3.1904]
avg_mistral = [3.4888, 3.2668, 3.7408, 3.7938, 3.8268]
avg_llama = [4.0164, 3.2988, 3.8994, 3.5166, 3.6028]

# Combine data into a list
data = [avg_gemma, avg_falcon, avg_qwen, avg_mistral, avg_llama]
model_names = ['Gemma', 'Falcon', 'Qwen', 'Mistral', 'LLaMA']
colors = ['#FF9999', '#66B2FF', '#99FF99', '#FFCC99', '#FF99CC']  # TBE (red), IBE (blue), then others

# Compute angle for each category
angles = [n / float(num_vars) * 2 * pi for n in range(num_vars)]
angles += angles[:1]  # Complete the loop

# Initialize the radar chart
fig, ax = plt.subplots(figsize=(6, 6), subplot_kw=dict(polar=True))

# Draw one axe per variable and add labels
ax.set_theta_offset(pi / 2)
ax.set_theta_direction(-1)

# Draw axis labels
plt.xticks(angles[:-1], labels,color="black", size=10, weight='bold')

# Remove default circular grid
ax.grid(False)

# Draw pentagon boundaries manually
for r in range(1, 6):
    pentagon = [r] * num_vars  # Constant radius for each level
    pentagon = np.append(pentagon, pentagon[0])  # Complete the loop
    ax.plot(angles, pentagon, color='gray', linestyle='--', linewidth=2)

# Draw bold dashed spokes (radial lines)
for angle in angles[:-1]:
    ax.plot([angle, angle], [0, 5], color='gray', linestyle='--', linewidth=1)

# Draw ylabels (scale) with bold and larger font
ax.set_rlabel_position(0)
plt.yticks([1, 2, 3, 4, 5], ["1", "2", "3", "4", "5"], color="black", size=10, weight='bold')
plt.ylim(0, 5)

# Plot each model's data
for i, model_data in enumerate(data):
    values = model_data + model_data[:1]  # Complete the loop
    # Plot the line and fill
    ax.plot(angles, values, linewidth=2, linestyle='solid', label=model_names[i], color=colors[i])
    ax.fill(angles, values, color=colors[i], alpha=0.25)
    # Add bold points where the line touches the axes
    for j in range(num_vars):
        ax.plot(angles[j], model_data[j], marker='o', markersize=10, color=colors[i])

# Add a legend
plt.legend(loc='upper right', bbox_to_anchor=(0.1, 0.1))

# Remove the outer circle (spine)
ax.spines['polar'].set_visible(False)

# Save the plot
plt.savefig('radar_chart_average_data.png')
plt.savefig('radar_chart_average_data_rawfc.pdf', format='pdf', bbox_inches='tight')

### lair_raw  falcon

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from math import pi

# Labels for criteria
labels = ['Informativeness', 'Accuracy', 'Readability', 'Objectivity', 'Logicality']
num_vars = len(labels)

# Updated data: FALCON → {Gemma, Falcon, Qwen, Mistral, LLaMA}
falcon_gemma   = [3.712, 2.889, 4.212, 3.695, 3.405]
falcon_falcon  = [4.292, 3.955, 3.564, 4.178, 4.178]
falcon_qwen    = [3.128, 2.9,   4.097, 3.883, 3.690]
falcon_mistral = [4.360, 3.491, 4.031, 4.132, 4.069]
falcon_llama   = [4.733, 3.923, 4.914, 4.595, 4.684]

# Combine data
data = [falcon_gemma, falcon_falcon, falcon_qwen, falcon_mistral, falcon_llama]
model_names = ['Gemma', 'Falcon', 'Qwen', 'Mistral', 'LLaMA']
colors = ['#FF9999', '#66B2FF', '#99FF99', '#FFCC99', '#FF99CC']

# Angles for radar chart
angles = [n / float(num_vars) * 2 * pi for n in range(num_vars)]
angles += angles[:1]  # close the circle

# Radar chart setup
fig, ax = plt.subplots(figsize=(6, 6), subplot_kw=dict(polar=True))
ax.set_theta_offset(pi / 2)
ax.set_theta_direction(-1)

# Category labels
plt.xticks(angles[:-1], labels, color="black", size=10, weight='bold')

# Remove grid
ax.grid(False)

# Draw concentric pentagon levels
for r in range(1, 6):
    pentagon = [r] * num_vars
    pentagon += pentagon[:1]
    ax.plot(angles, pentagon, color='gray', linestyle='--', linewidth=2)

# Draw radial lines
for angle in angles[:-1]:
    ax.plot([angle, angle], [0, 5], color='gray', linestyle='--', linewidth=1)

# Y-labels
ax.set_rlabel_position(0)
plt.yticks([1, 2, 3, 4, 5], ["1", "2", "3", "4", "5"], color="black", size=10, weight='bold')
plt.ylim(0, 5)

# Plot and fill each model's data
for i, model_data in enumerate(data):
    values = model_data + model_data[:1]
    ax.plot(angles, values, linewidth=2, linestyle='solid', label=model_names[i], color=colors[i])
    ax.fill(angles, values, color=colors[i], alpha=0.25)
    for j in range(num_vars):
        ax.plot(angles[j], model_data[j], marker='o', markersize=10, color=colors[i])

# Legend
plt.legend(loc='upper right', bbox_to_anchor=(0.1, 0.1))

# Remove outer polar spine
ax.spines['polar'].set_visible(False)

# Save outputs
plt.savefig('radar_chart_falcon_data.png')
plt.savefig('radar_chart_falcon_data_lair_raw.pdf', format='pdf', bbox_inches='tight')


## qwen lair_raw

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from math import pi

# Labels for criteria
labels = ['Informativeness', 'Accuracy', 'Readability', 'Objectivity', 'Logicality']
num_vars = len(labels)

# Updated QWEN → {Gemma, Falcon, Qwen, Mistral, LLaMA}
qwen_gemma   = [3.814, 3.111, 4.327, 3.889, 3.624]
qwen_falcon  = [4.217, 3.843, 3.496, 4.092, 4.183]
qwen_qwen    = [3.177, 2.817, 4.091, 3.646, 3.563]
qwen_mistral = [4.307, 3.430, 4.100, 4.117, 4.211]
qwen_llama   = [4.731, 3.816, 4.767, 4.445, 4.537]

# Combine data
data = [qwen_gemma, qwen_falcon, qwen_qwen, qwen_mistral, qwen_llama]
model_names = ['Gemma', 'Falcon', 'Qwen', 'Mistral', 'LLaMA']
colors = ['#FF9999', '#66B2FF', '#99FF99', '#FFCC99', '#FF99CC']

# Radar chart angles
angles = [n / float(num_vars) * 2 * pi for n in range(num_vars)]
angles += angles[:1]

# Radar chart setup
fig, ax = plt.subplots(figsize=(6, 6), subplot_kw=dict(polar=True))
ax.set_theta_offset(pi / 2)
ax.set_theta_direction(-1)

# Criteria axis labels
plt.xticks(angles[:-1], labels, color="black", size=10, weight='bold')

# Remove default grid
ax.grid(False)

# Draw concentric pentagon levels
for r in range(1, 6):
    pentagon = [r] * num_vars + [r]
    ax.plot(angles, pentagon, color='gray', linestyle='--', linewidth=2)

# Radial lines
for angle in angles[:-1]:
    ax.plot([angle, angle], [0, 5], color='gray', linestyle='--', linewidth=1)

# Y-axis ticks
ax.set_rlabel_position(0)
plt.yticks([1, 2, 3, 4, 5], ["1", "2", "3", "4", "5"], color="black", size=10, weight='bold')
plt.ylim(0, 5)

# Plot each model's data
for i, model_data in enumerate(data):
    values = model_data + model_data[:1]
    ax.plot(angles, values, linewidth=2, linestyle='solid', label=model_names[i], color=colors[i])
    ax.fill(angles, values, color=colors[i], alpha=0.25)
    for j in range(num_vars):
        ax.plot(angles[j], model_data[j], marker='o', markersize=10, color=colors[i])

# Legend
plt.legend(loc='upper right', bbox_to_anchor=(0.1, 0.1))

# Remove polar spine
ax.spines['polar'].set_visible(False)

# Save outputs
plt.savefig('radar_chart_qwen_data.png')
plt.savefig('radar_chart_qwen_data_lair_raw.pdf', format='pdf', bbox_inches='tight')


## mistral lair_raw

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from math import pi

# Labels for evaluation criteria
labels = ['Informativeness', 'Accuracy', 'Readability', 'Objectivity', 'Logicality']
num_vars = len(labels)

# Updated MISTRAL → {Gemma, Falcon, Qwen, Mistral, LLaMA}
mistral_gemma   = [3.667, 2.94,  4.195, 3.724, 3.403]
mistral_falcon  = [4.151, 3.832, 3.393, 4.119, 4.160]
mistral_qwen    = [3.115, 2.868, 4.044, 3.754, 3.566]
mistral_mistral = [4.298, 3.472, 4.070, 4.182, 4.141]
mistral_llama   = [4.779, 3.933, 4.786, 4.545, 4.624]

# Combine into dataset
data = [mistral_gemma, mistral_falcon, mistral_qwen, mistral_mistral, mistral_llama]
model_names = ['Gemma', 'Falcon', 'Qwen', 'Mistral', 'LLaMA']
colors = ['#FF9999', '#66B2FF', '#99FF99', '#FFCC99', '#FF99CC']

# Calculate radar chart angles
angles = [n / float(num_vars) * 2 * pi for n in range(num_vars)]
angles += angles[:1]

# Initialize radar chart
fig, ax = plt.subplots(figsize=(6, 6), subplot_kw=dict(polar=True))
ax.set_theta_offset(pi / 2)
ax.set_theta_direction(-1)

# Add category labels
plt.xticks(angles[:-1], labels, color="black", size=10, weight='bold')

# Remove default grid
ax.grid(False)

# Draw pentagon boundaries
for r in range(1, 6):
    pentagon = [r] * num_vars + [r]
    ax.plot(angles, pentagon, color='gray', linestyle='--', linewidth=2)

# Draw radial lines
for angle in angles[:-1]:
    ax.plot([angle, angle], [0, 5], color='gray', linestyle='--', linewidth=1)

# Add radial axis labels
ax.set_rlabel_position(0)
plt.yticks([1, 2, 3, 4, 5], ["1", "2", "3", "4", "5"], color="black", size=10, weight='bold')
plt.ylim(0, 5)

# Plot each model’s radar line
for i, model_data in enumerate(data):
    values = model_data + model_data[:1]
    ax.plot(angles, values, linewidth=2, linestyle='solid', label=model_names[i], color=colors[i])
    ax.fill(angles, values, color=colors[i], alpha=0.25)
    for j in range(num_vars):
        ax.plot(angles[j], model_data[j], marker='o', markersize=10, color=colors[i])

# Add legend
plt.legend(loc='upper right', bbox_to_anchor=(0.1, 0.1))

# Hide outer circular spine
ax.spines['polar'].set_visible(False)

# Save figure
plt.savefig('radar_chart_mistral_data.png')
plt.savefig('radar_chart_mistral_data_lair_raw.pdf', format='pdf', bbox_inches='tight')


### llama 

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from math import pi

# Labels for evaluation criteria
labels = ['Informativeness', 'Accuracy', 'Readability', 'Objectivity', 'Logicality']
num_vars = len(labels)

# Updated LLAMA → {Gemma, Falcon, Qwen, Mistral, LLaMA}
llama_gemma   = [3.514, 2.942, 4.058, 3.787, 3.514]
llama_falcon  = [3.884, 3.797, 3.483, 4.003, 3.991]
llama_qwen    = [2.966, 2.816, 3.948, 3.634, 3.493]
llama_mistral = [4.131, 3.444, 4.049, 4.144, 4.131]
llama_llama   = [4.521, 3.713, 4.511, 4.170, 4.263]

# Combine into dataset
data = [llama_gemma, llama_falcon, llama_qwen, llama_mistral, llama_llama]
model_names = ['Gemma', 'Falcon', 'Qwen', 'Mistral', 'LLaMA']
colors = ['#FF9999', '#66B2FF', '#99FF99', '#FFCC99', '#FF99CC']

# Calculate radar chart angles
angles = [n / float(num_vars) * 2 * pi for n in range(num_vars)]
angles += angles[:1]

# Initialize radar chart
fig, ax = plt.subplots(figsize=(6, 6), subplot_kw=dict(polar=True))
ax.set_theta_offset(pi / 2)
ax.set_theta_direction(-1)

# Add category labels
plt.xticks(angles[:-1], labels, color="black", size=10, weight='bold')

# Remove default grid
ax.grid(False)

# Draw pentagon boundaries
for r in range(1, 6):
    pentagon = [r] * num_vars + [r]
    ax.plot(angles, pentagon, color='gray', linestyle='--', linewidth=2)

# Draw radial lines
for angle in angles[:-1]:
    ax.plot([angle, angle], [0, 5], color='gray', linestyle='--', linewidth=1)

# Add radial axis labels
ax.set_rlabel_position(0)
plt.yticks([1, 2, 3, 4, 5], ["1", "2", "3", "4", "5"], color="black", size=10, weight='bold')
plt.ylim(0, 5)

# Plot each model’s radar line
for i, model_data in enumerate(data):
    values = model_data + model_data[:1]
    ax.plot(angles, values, linewidth=2, linestyle='solid', label=model_names[i], color=colors[i])
    ax.fill(angles, values, color=colors[i], alpha=0.25)
    for j in range(num_vars):
        ax.plot(angles[j], model_data[j], marker='o', markersize=10, color=colors[i])

# Add legend
plt.legend(loc='upper right', bbox_to_anchor=(0.1, 0.1))

# Hide outer circular spine
ax.spines['polar'].set_visible(False)

# Save figure
plt.savefig('radar_chart_llama_data.png')
plt.savefig('radar_chart_llama_data_lair_raw.pdf', format='pdf', bbox_inches='tight')


## gemma

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from math import pi

# Labels for evaluation criteria
labels = ['Informativeness', 'Accuracy', 'Readability', 'Objectivity', 'Logicality']
num_vars = len(labels)

# Updated GEMMA → {Falcon, Qwen, Gemma, Mistral, LLaMA}
gemma_falcon  = [3.548, 3.548, 2.785, 3.616, 3.388]
gemma_qwen    = [2.488, 2.522, 3.305, 3.222, 3.027]
gemma_gemma   = [2.873, 2.599, 3.412, 3.561, 3.170]
gemma_mistral = [3.703, 3.482, 3.396, 3.909, 3.987]
gemma_llama   = [3.530, 2.824, 2.482, 2.954, 2.970]

# Combine into dataset
data = [gemma_falcon, gemma_qwen, gemma_gemma, gemma_mistral, gemma_llama]
model_names = ['Falcon', 'Qwen', 'Gemma', 'Mistral', 'LLaMA']
colors = ['#66B2FF', '#99FF99', '#FF9999', '#FFCC99', '#FF99CC']  # consistent color scheme

# Calculate radar chart angles
angles = [n / float(num_vars) * 2 * pi for n in range(num_vars)]
angles += angles[:1]

# Initialize radar chart
fig, ax = plt.subplots(figsize=(6, 6), subplot_kw=dict(polar=True))
ax.set_theta_offset(pi / 2)
ax.set_theta_direction(-1)

# Add category labels
plt.xticks(angles[:-1], labels, color="black", size=10, weight='bold')

# Remove default grid
ax.grid(False)

# Draw pentagon boundaries
for r in range(1, 6):
    pentagon = [r] * num_vars + [r]
    ax.plot(angles, pentagon, color='gray', linestyle='--', linewidth=2)

# Draw radial lines
for angle in angles[:-1]:
    ax.plot([angle, angle], [0, 5], color='gray', linestyle='--', linewidth=1)

# Add radial axis labels
ax.set_rlabel_position(0)
plt.yticks([1, 2, 3, 4, 5], ["1", "2", "3", "4", "5"], color="black", size=10, weight='bold')
plt.ylim(0, 5)

# Plot each model’s radar line
for i, model_data in enumerate(data):
    values = model_data + model_data[:1]
    ax.plot(angles, values, linewidth=2, linestyle='solid', label=model_names[i], color=colors[i])
    ax.fill(angles, values, color=colors[i], alpha=0.25)
    for j in range(num_vars):
        ax.plot(angles[j], model_data[j], marker='o', markersize=10, color=colors[i])

# Add legend
plt.legend(loc='upper right', bbox_to_anchor=(0.1, 0.1))

# Hide outer circular spine
ax.spines['polar'].set_visible(False)

# Save figure
plt.savefig('radar_chart_gemma_data.png')
plt.savefig('radar_chart_gemma_data_lair_raw.pdf', format='pdf', bbox_inches='tight')


## average

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from math import pi

# Labels for evaluation criteria
labels = ['Informativeness', 'Accuracy', 'Readability', 'Objectivity', 'Logicality']
num_vars = len(labels)

# Average scores for each model
avg_gemma   = [3.5160, 2.8962, 4.0408, 3.7312, 3.4232]
avg_falcon  = [4.0184, 3.7950, 3.3442, 4.0016, 3.9800]
avg_qwen    = [2.9748, 2.7846, 3.8970, 3.6278, 3.4678]
avg_mistral = [4.1598, 3.4638, 3.9292, 4.0968, 4.1078]
avg_llama   = [4.4588, 3.6418, 4.2920, 4.1418, 4.2156]

# Combine into dataset
data = [avg_gemma, avg_falcon, avg_qwen, avg_mistral, avg_llama]
model_names = ['Gemma', 'Falcon', 'Qwen', 'Mistral', 'LLaMA']
colors = ['#FF9999', '#66B2FF', '#99FF99', '#FFCC99', '#FF99CC']  # consistent color scheme

# Calculate radar chart angles
angles = [n / float(num_vars) * 2 * pi for n in range(num_vars)]
angles += angles[:1]

# Initialize radar chart
fig, ax = plt.subplots(figsize=(6, 6), subplot_kw=dict(polar=True))
ax.set_theta_offset(pi / 2)
ax.set_theta_direction(-1)

# Add category labels
plt.xticks(angles[:-1], labels, color="black", size=10, weight='bold')

# Remove default grid
ax.grid(False)

# Draw pentagon boundaries
for r in range(1, 6):
    pentagon = [r] * num_vars + [r]
    ax.plot(angles, pentagon, color='gray', linestyle='--', linewidth=2)

# Draw radial lines
for angle in angles[:-1]:
    ax.plot([angle, angle], [0, 5], color='gray', linestyle='--', linewidth=1)

# Add radial axis labels
ax.set_rlabel_position(0)
plt.yticks([1, 2, 3, 4, 5], ["1", "2", "3", "4", "5"], color="black", size=10, weight='bold')
plt.ylim(0, 5)

# Plot each model’s radar line
for i, model_data in enumerate(data):
    values = model_data + model_data[:1]
    ax.plot(angles, values, linewidth=2, linestyle='solid', label=model_names[i], color=colors[i])
    ax.fill(angles, values, color=colors[i], alpha=0.25)
    for j in range(num_vars):
        ax.plot(angles[j], model_data[j], marker='o', markersize=10, color=colors[i])

# Add legend
plt.legend(loc='upper right', bbox_to_anchor=(0.1, 0.1))

# Hide outer circular spine
ax.spines['polar'].set_visible(False)

# Save figure
plt.savefig('radar_chart_model_avg_scores.png')
plt.savefig('radar_chart_model_avg_scores_lair_raw.pdf', format='pdf', bbox_inches='tight')


## confusion matrix for the rawfc 

In [ ]:
import json
from sklearn.metrics import confusion_matrix
import pandas as pd

def load_data(path):
    """Load JSON prediction data from the given file path."""
    with open(path, 'r', encoding='utf-8') as f:
        return json.load(f)

def compute_confusion_matrix(data, labels):
    """Compute a confusion matrix given a list of dicts with 'true_label' and 'predicted_label'."""
    y_true = [item['true_label']      for item in data]
    y_pred = [item['predicted_label'] for item in data]
    return confusion_matrix(y_true, y_pred, labels=labels)

def main():
    # Map dataset names to their JSON file paths
    files = {
        "TB2_llama_lora_plus_42": "/data2/Gaurav/retrieve/rawfc/error_analysis_rawfc/rawfc_prediction_for_confusion_matric/TB2_llama_lora_plus_42predictions_with_responses.json",
        "TBE1_lora_123_qwen":     "/data2/Gaurav/retrieve/rawfc/error_analysis_rawfc/rawfc_prediction_for_confusion_matric/TBE1_lora_123_qwen_predictions_with_responses.json",
        "TBE3_rawfc_llama_123":   "/data2/Gaurav/retrieve/rawfc/error_analysis_rawfc/rawfc_prediction_for_confusion_matric/TBE3_rawfc_predictions_llama_123.json"
    }
    
    # Define the label ordering for the matrix
    labels = ["true", "false", "half"]
    
    for name, path in files.items():
        data = load_data(path)
        cm = compute_confusion_matrix(data, labels)
        
        # Wrap in a DataFrame for nicer printing
        df_cm = pd.DataFrame(
            cm,
            index=[f"True: {lbl}"  for lbl in labels],
            columns=[f"Pred: {lbl}" for lbl in labels]
        )
        
        print(f"\nConfusion Matrix for {name}:\n")
        print(df_cm)
        print("-" * 50)

if __name__ == "__main__":
    main()


## IBES

In [ ]:
import json
from sklearn.metrics import confusion_matrix
import pandas as pd
import re

def load_json(path):
    """Load JSON data from a file."""
    with open(path, 'r', encoding='utf-8') as f:
        return json.load(f)

def extract_labels(item):
    """
    Extract actual and predicted labels from a single record.
    Supports keys: 'actual_label', 'label', 'predicted_label', and parsing from 'model_response'.
    """
    actual = item.get('actual_label') or item.get('label')
    pred   = item.get('predicted_label')
    if not pred and 'model_response' in item:
        match = re.search(r'Claim Veracity:\s*(\w+)', item['model_response'])
        if match:
            pred = match.group(1)
    return actual, pred

def compute_and_display_cm(data, name, label_order=None):
    """
    Compute and display the confusion matrix for a dataset.
    - data: list of dict records
    - name: identifier for printing
    - label_order: optional list specifying label ordering
    """
    y_true, y_pred = [], []
    for rec in data:
        actual, pred = extract_labels(rec)
        if actual is not None and pred is not None:
            y_true.append(actual)
            y_pred.append(pred)
    
    if not y_true:
        print(f"No valid label pairs for '{name}'. Skipping.\n")
        return
    
    # Determine the label set/order
    labels = label_order or sorted(set(y_true) | set(y_pred))
    
    # Build and print confusion matrix
    cm = confusion_matrix(y_true, y_pred, labels=labels)
    df_cm = pd.DataFrame(
        cm,
        index=[f"True: {lbl}" for lbl in labels],
        columns=[f"Pred: {lbl}" for lbl in labels]
    )
    print(f"=== Confusion Matrix for {name} ===\n")
    print(df_cm, "\n")

def main():
    # Update these paths to wherever your files actually live
    files = {
        "IBE1_qwen":         "rawfc_prediction_for_confusion_matric/IBE1qwen_generated_results.json",
        "IBE2_meta_llama":   "rawfc_prediction_for_confusion_matric/IBE2meta_llama_generated_results.json",
        "IBE3_understanding":"rawfc_prediction_for_confusion_matric/IBE3understanding_test_qwen.json",
        "IBE4_qwen":         "rawfc_prediction_for_confusion_matric/IBE4qwen_generated_results.json"
    }
    
    # If you want a consistent row/column order in every matrix:
    label_order = ["true", "half", "false"]
    
    for name, path in files.items():
        data = load_json(path)
        compute_and_display_cm(data, name, label_order=label_order)

if __name__ == "__main__":
    main()


In [ ]:
import json
import re
from sklearn.metrics import confusion_matrix
import pandas as pd
import matplotlib.pyplot as plt

# 1. Define your datasets in the order you want them grouped
dataset_names = ["TBE1", "TBE2", "TBE3", "IBE1", "IBE2", "IBE3", "IBE4"]
paths = {
    "TBE1": "rawfc_prediction_for_confusion_matric/TBE1_lora_123_qwen_predictions_with_responses.json",
    "TBE2": "rawfc_prediction_for_confusion_matric/TB2_llama_lora_plus_42predictions_with_responses.json",
    "TBE3": "rawfc_prediction_for_confusion_matric/TBE3_rawfc_predictions_llama_123.json",
    "IBE1": "rawfc_prediction_for_confusion_matric/IBE1qwen_generated_results.json",
    "IBE2": "rawfc_prediction_for_confusion_matric/IBE2meta_llama_generated_results.json",
    "IBE3": "rawfc_prediction_for_confusion_matric/IBE3understanding_test_qwen.json",
    "IBE4": "rawfc_prediction_for_confusion_matric/IBE4qwen_generated_results.json"
}

# 2. Specify the label ordering
label_order = ["true", "false", "half"]
col_labels  = ["TRUE", "FALSE", "half"]

# 3. Helper to extract true/pred labels
def extract_labels(item):
    actual = item.get("true_label") or item.get("actual_label") or item.get("label")
    pred   = item.get("predicted_label")
    if not pred and "model_response" in item:
        m = re.search(r"Claim Veracity:\s*(\w+)", item["model_response"])
        if m:
            pred = m.group(1)
    return actual, pred

# 4. Load each dataset and compute its confusion matrix
cms = {}
for name in dataset_names:
    data = json.load(open(paths[name], "r", encoding="utf-8"))
    y_true, y_pred = [], []
    for rec in data:
        a, p = extract_labels(rec)
        if a and p:
            y_true.append(a)
            y_pred.append(p)
    cms[name] = confusion_matrix(y_true, y_pred, labels=label_order)

# 5. Build a DataFrame with rows grouped by actual label first
rows = []
index = []
for act_idx, act in enumerate(label_order):
    for name in dataset_names:
        rows.append(cms[name][act_idx].tolist())
        index.append((act.upper(), name))

multi_idx = pd.MultiIndex.from_tuples(index, names=("ACTUAL", "DATASET"))
df = pd.DataFrame(rows, index=multi_idx, columns=col_labels)

# 6. Plot the single combined heatmap
plt.figure(figsize=(6, 10))
plt.imshow(df.values, aspect="auto")
plt.xticks(range(len(col_labels)), col_labels, fontsize=12)
yticks = [f"{act} – {ds}" for act, ds in df.index]
plt.yticks(range(len(yticks)), yticks, fontsize=10)
for i in range(df.shape[0]):
    for j in range(df.shape[1]):
        plt.text(j, i, df.values[i, j], ha="center", va="center")
plt.xlabel("Predicted Label", fontsize=14)
plt.title("Combined Confusion Matrix (grouped by ACTUAL)", fontsize=16)
plt.tight_layout()
plt.show()


In [ ]:
import json
import re
import numpy as np
import matplotlib.pyplot as plt
from sklearn.metrics import confusion_matrix

# 1. Define your datasets and file paths
dataset_names = ["TBE1", "TBE2", "TBE3", "IBE1", "IBE2", "IBE3", "IBE4"]
paths = {
    "TBE1": "rawfc_prediction_for_confusion_matric/TBE1_lora_123_qwen_predictions_with_responses.json",
    "TBE2": "rawfc_prediction_for_confusion_matric/TB2_llama_lora_plus_42predictions_with_responses.json",
    "TBE3": "rawfc_prediction_for_confusion_matric/TBE3_rawfc_predictions_llama_123.json",
    "IBE1": "rawfc_prediction_for_confusion_matric/IBE1qwen_generated_results.json",
    "IBE2": "rawfc_prediction_for_confusion_matric/IBE2meta_llama_generated_results.json",
    "IBE3": "rawfc_prediction_for_confusion_matric/IBE3understanding_test_qwen.json",
    "IBE4": "rawfc_prediction_for_confusion_matric/IBE4qwen_generated_results.json"
}

# 2. Specify the label order
label_order = ["true", "false", "half"]
col_labels = ["TRUE", "FALSE", "half"]

# 3. Helper to extract actual and predicted labels
def extract_labels(item):
    actual = item.get("true_label") or item.get("actual_label") or item.get("label")
    pred   = item.get("predicted_label")
    if not pred and "model_response" in item:
        m = re.search(r"Claim Veracity:\s*(\w+)", item["model_response"])
        if m:
            pred = m.group(1)
    return actual, pred

# 4. Compute a confusion matrix for each dataset
cms = []
for ds in dataset_names:
    data = json.load(open(paths[ds], "r", encoding="utf-8"))
    y_true, y_pred = [], []
    for rec in data:
        a, p = extract_labels(rec)
        if a and p:
            y_true.append(a)
            y_pred.append(p)
    cms.append(confusion_matrix(y_true, y_pred, labels=label_order))

# 5. Stack rows: all TRUE rows, then all FALSE, then all HALF
n = len(dataset_names)
rows = []
for i in range(len(label_order)):
    for cm in cms:
        rows.append(cm[i])
arr = np.vstack(rows)  # shape = (3*n, 3)

# 6. Plot single combined heatmap

fig, ax = plt.subplots(figsize=(6, 8))
im = ax.imshow(arr, aspect="auto")

# -- Predicted labels on top
ax.set_xticks(np.arange(len(col_labels)))
ax.set_xticklabels(col_labels, fontsize=12)
ax.xaxis.set_label_position("top")
ax.xaxis.tick_top()
ax.set_xlabel("Predicted Label", fontsize=14, labelpad=10)

# -- Grouped Actual labels on the left
group_positions = [n/2 - 0.5, n + n/2 - 0.5, 2*n + n/2 - 0.5]
group_labels = [lbl.upper() for lbl in label_order]
ax.set_yticks(group_positions)
ax.set_yticklabels(group_labels, fontsize=12)
ax.set_ylabel("Actual Label", fontsize=14, labelpad=10)

# -- Draw horizontal separators between the groups
for sep in [n - 0.5, 2*n - 0.5]:
    ax.hlines(sep, -0.5, len(col_labels)-0.5, colors="white", linewidth=2)

# -- Annotate each cell with the count
for i in range(arr.shape[0]):
    for j in range(arr.shape[1]):
        ax.text(j, i, int(arr[i, j]), ha="center", va="center", color="white")

plt.tight_layout()
plt.show()


In [ ]:
import json
import re
import numpy as np
import matplotlib.pyplot as plt
from sklearn.metrics import confusion_matrix

# 1. Define datasets and file paths
dataset_names = ["TBE1", "TBE2", "TBE3", "IBE1", "IBE2", "IBE3", "IBE4"]
paths = {
    "TBE1": "rawfc_prediction_for_confusion_matric/TBE1_lora_123_qwen_predictions_with_responses.json",
    "TBE2": "rawfc_prediction_for_confusion_matric/TB2_llama_lora_plus_42predictions_with_responses.json",
    "TBE3": "rawfc_prediction_for_confusion_matric/TBE3_rawfc_predictions_llama_123.json",
    "IBE1": "rawfc_prediction_for_confusion_matric/IBE1qwen_generated_results.json",
    "IBE2": "rawfc_prediction_for_confusion_matric/IBE2meta_llama_generated_results.json",
    "IBE3": "rawfc_prediction_for_confusion_matric/IBE3understanding_test_qwen.json",
    "IBE4": "rawfc_prediction_for_confusion_matric/IBE4qwen_generated_results.json"
}

# 2. Define label ordering
label_order = ["true", "false", "half"]
col_labels = ["TRUE", "FALSE", "half"]

# 3. Helper to extract actual & predicted labels
def extract_labels(item):
    actual = item.get("true_label") or item.get("actual_label") or item.get("label")
    pred = item.get("predicted_label")
    if not pred and "model_response" in item:
        m = re.search(r"Claim Veracity:\s*(\w+)", item["model_response"])
        if m:
            pred = m.group(1)
    return actual, pred

# 4. Compute a confusion matrix for each dataset
cms = []
for ds in dataset_names:
    data = json.load(open(paths[ds], "r", encoding="utf-8"))
    y_true, y_pred = [], []
    for rec in data:
        a, p = extract_labels(rec)
        if a and p:
            y_true.append(a)
            y_pred.append(p)
    cms.append(confusion_matrix(y_true, y_pred, labels=label_order))

# 5. Stack rows: all TRUE rows, then FALSE, then HALF
n = len(dataset_names)
rows = []
for i in range(len(label_order)):
    for cm in cms:
        rows.append(cm[i])
arr = np.vstack(rows)  # shape = (3*n, 3)

# 6. Plot combined heatmap
fig, ax = plt.subplots(figsize=(8, 10))

# show heatmap
im = ax.imshow(arr, aspect="auto")

# Predicted labels on top
ax.set_xticks(np.arange(len(col_labels)))
ax.set_xticklabels(col_labels, fontsize=12)
ax.xaxis.set_label_position("top")
ax.xaxis.tick_top()
ax.set_xlabel("Predicted Label", fontsize=14, labelpad=10)

# Dataset names per row
ylabels = dataset_names * len(label_order)
ax.set_yticks(np.arange(len(ylabels)))
ax.set_yticklabels(ylabels, fontsize=10)

# Draw separators between actual-label groups
for sep in [n - 0.5, 2 * n - 0.5]:
    ax.axhline(sep, color="white", linewidth=2)

# Annotate counts
for i in range(arr.shape[0]):
    for j in range(arr.shape[1]):
        ax.text(j, i, int(arr[i, j]), ha="center", va="center", color="white")

# Add one "TRUE", "FALSE", "HALF" label to the left of each block
for idx, lbl in enumerate(label_order):
    center = idx * n + (n - 1) / 2
    ax.text(-1.2, center, lbl.upper(), ha="center", va="center", fontsize=12)

plt.tight_layout()
plt.show()


In [ ]:
import json
import re
import numpy as np
import matplotlib.pyplot as plt
from sklearn.metrics import confusion_matrix

# 1. Define datasets and file paths
dataset_names = ["TBE1", "TBE2", "TBE3", "IBE1", "IBE2", "IBE3", "IBE4"]
paths = {
    "TBE1": "rawfc_prediction_for_confusion_matric/TBE1_lora_123_qwen_predictions_with_responses.json",
    "TBE2": "rawfc_prediction_for_confusion_matric/TB2_llama_lora_plus_42predictions_with_responses.json",
    "TBE3": "rawfc_prediction_for_confusion_matric/TBE3_rawfc_predictions_llama_123.json",
    "IBE1": "rawfc_prediction_for_confusion_matric/IBE1qwen_generated_results.json",
    "IBE2": "rawfc_prediction_for_confusion_matric/IBE2meta_llama_generated_results.json",
    "IBE3": "rawfc_prediction_for_confusion_matric/IBE3understanding_test_qwen.json",
    "IBE4": "rawfc_prediction_for_confusion_matric/IBE4qwen_generated_results.json"
}


# 2. Define label ordering
label_order = ["true", "false", "half"]
col_labels  = ["true", "false", "half"]

# 3. Helper: extract actual & predicted labels
def extract_labels(item):
    actual = item.get("true_label") or item.get("actual_label") or item.get("label")
    pred   = item.get("predicted_label")
    if not pred and "model_response" in item:
        m = re.search(r"Claim Veracity:\s*(\w+)", item["model_response"])
        if m:
            pred = m.group(1)
    return actual, pred

# 4. Compute confusion matrices for each dataset
cms = []
for ds in dataset_names:
    with open(paths[ds], "r", encoding="utf-8") as f:
        data = json.load(f)
    y_true, y_pred = [], []
    for rec in data:
        a, p = extract_labels(rec)
        if a and p:
            y_true.append(a)
            y_pred.append(p)
    cms.append(confusion_matrix(y_true, y_pred, labels=label_order))

# 5. Stack rows: all TRUE rows, then FALSE, then HALF
n = len(dataset_names)
rows = []
for i in range(3):
    for cm in cms:
        rows.append(cm[i])
arr = np.vstack(rows)  # shape = (3*n, 3)

# 6. Plot heatmap and save as PDF
fig, ax = plt.subplots(figsize=(8, 10))
im = ax.imshow(arr, aspect="auto")

# Colorbar explaining which color maps to which count
cbar = fig.colorbar(im, ax=ax)
cbar.set_label("Count", rotation=270, labelpad=15)

# Predicted labels at top
ax.set_xticks(np.arange(len(col_labels)))
ax.set_xticklabels(col_labels, fontsize=12)
ax.xaxis.set_label_position("top")
ax.xaxis.tick_top()
ax.set_xlabel("Predicted Label", fontsize=14, labelpad=10)

# Dataset names on each row
ylabels = dataset_names * 3
ax.set_yticks(np.arange(len(ylabels)))
ax.set_yticklabels(ylabels, fontsize=10)

# Horizontal separators between TRUE/FALSE/HALF blocks
for sep in [n - 0.5, 2*n - 0.5]:
    ax.axhline(sep, color="white", linewidth=2)

# Annotate each cell with its count
for i in range(arr.shape[0]):
    for j in range(arr.shape[1]):
        ax.text(j, i, int(arr[i, j]), ha="center", va="center", color="white")

# Block labels ("TRUE","FALSE","HALF") on the left
for idx, lbl in enumerate(["true", "false", "half"]):
    center = idx * n + (n - 1) / 2
    ax.text(-1.5, center, lbl, ha="center", va="center", fontsize=12)

plt.tight_layout()
output_pdf = "combined_confusion_matrix_rawfc.pdf"
fig.savefig(output_pdf, format="pdf", bbox_inches="tight")
print(f"Saved combined confusion matrix to {output_pdf}")


In [ ]:
import json
import re
import numpy as np
import matplotlib.pyplot as plt
from sklearn.metrics import confusion_matrix

# 1. Define datasets and file paths
dataset_names = ["TBE1", "TBE2", "TBE3", "IBE1", "IBE2", "IBE3", "IBE4"]
paths = {
    "TBE1": "rawfc_prediction_for_confusion_matric/TBE1_lora_123_qwen_predictions_with_responses.json",
    "TBE2": "rawfc_prediction_for_confusion_matric/TB2_llama_lora_plus_42predictions_with_responses.json",
    "TBE3": "rawfc_prediction_for_confusion_matric/TBE3_rawfc_predictions_llama_123.json",
    "IBE1": "rawfc_prediction_for_confusion_matric/IBE1qwen_generated_results.json",
    "IBE2": "rawfc_prediction_for_confusion_matric/IBE2meta_llama_generated_results.json",
    "IBE3": "rawfc_prediction_for_confusion_matric/IBE3understanding_test_qwen.json",
    "IBE4": "rawfc_prediction_for_confusion_matric/IBE4qwen_generated_results.json"
}

# 2. Define label ordering
label_order = ["true", "false", "half"]
col_labels  = ["true", "false", "half"]

# 3. Helper: extract actual & predicted labels
def extract_labels(item):
    actual = item.get("true_label") or item.get("actual_label") or item.get("label")
    pred   = item.get("predicted_label")
    if not pred and "model_response" in item:
        m = re.search(r"Claim Veracity:\s*(\w+)", item["model_response"])
        if m:
            pred = m.group(1)
    return actual, pred

# 4. Compute confusion matrices for each dataset
cms = []
for ds in dataset_names:
    with open(paths[ds], "r", encoding="utf-8") as f:
        data = json.load(f)
    y_true, y_pred = [], []
    for rec in data:
        a, p = extract_labels(rec)
        if a and p:
            y_true.append(a)
            y_pred.append(p)
    cms.append(confusion_matrix(y_true, y_pred, labels=label_order))

# 5. Stack rows: all TRUE rows, then FALSE, then HALF
n = len(dataset_names)
rows = []
for i in range(3):
    for cm in cms:
        rows.append(cm[i])
arr = np.vstack(rows)  # shape = (3*n, 3)

# 6. Plot heatmap and save as PDF
fig, ax = plt.subplots(figsize=(8, 10))
im = ax.imshow(arr, aspect="auto")

# Colorbar explaining which color maps to which count
cbar = fig.colorbar(im, ax=ax)
cbar.set_label("Count", rotation=270, labelpad=15)

# Predicted labels at top
ax.set_xticks(np.arange(len(col_labels)))
ax.set_xticklabels(col_labels, fontsize=12)
ax.xaxis.set_label_position("top")
ax.xaxis.tick_top()
ax.set_xlabel("Predicted Label", fontsize=14, labelpad=10)

# Dataset names on each row
ylabels = dataset_names * 3
ax.set_yticks(np.arange(len(ylabels)))
ax.set_yticklabels(ylabels, fontsize=10)

# Set y-axis label
ax.set_ylabel("Actual Label", fontsize=14, labelpad=10)

# Horizontal separators between TRUE/FALSE/HALF blocks
for sep in [n - 0.5, 2*n - 0.5]:
    ax.axhline(sep, color="white", linewidth=2)

# Annotate each cell with its count
for i in range(arr.shape[0]):
    for j in range(arr.shape[1]):
        ax.text(j, i, int(arr[i, j]), ha="center", va="center", color="white")

# Block labels ("TRUE","FALSE","HALF") on the left
for idx, lbl in enumerate(["true", "false", "half"]):
    center = idx * n + (n - 1) / 2
    ax.text(-1.5, center, lbl, ha="center", va="center", fontsize=12)

plt.tight_layout()
output_pdf = "combined_confusion_matrix_rawfc.pdf"
fig.savefig(output_pdf, format="pdf", bbox_inches="tight")
print(f"Saved combined confusion matrix to {output_pdf}")

## lair raw

In [ ]:
import json
import re
import numpy as np
import matplotlib.pyplot as plt
from sklearn.metrics import confusion_matrix

# 1. Define datasets and file paths
dataset_names = ["TBE1", "TBE2", "TBE3", "IBE1", "IBE2", "IBE3", "IBE4"]
paths = {
    "TBE1": "lairraw_prediction_for_confusion_matric/TBE1_lora_llama_predictions_detailed.json",
    "TBE2": "lairraw_prediction_for_confusion_matric/TBE2_lora_mistral_predictions_with_responses.json",
    "TBE3": "lairraw_prediction_for_confusion_matric/TBE3_lair_raw_xlnet_llama_seed42_predictions.json",
    "IBE1": "lairraw_prediction_for_confusion_matric/IBE1mistral_generated_results.json",
    "IBE2": "lairraw_prediction_for_confusion_matric/IBE2meta_llama_generated_results.json",
    "IBE3": "lairraw_prediction_for_confusion_matric/IBE3meta_llama_generated_results.json",
    "IBE4": "lairraw_prediction_for_confusion_matric/IBE4mistral_generated_results.json"
}

# 2. Define the six-label ordering
label_order = [
    "true",
    "false",
    "half-true",
    "mostly-true",
    "barely-true",
    "pants-fire"
]
col_labels = label_order

# 3. Helper: extract actual & predicted labels
def extract_labels(item):
    actual = item.get("true_label") or item.get("actual_label") or item.get("label")
    pred   = item.get("predicted_label")
    if not pred and "model_response" in item:
        m = re.search(r"Claim Veracity:\s*([\w-]+)", item["model_response"])
        if m:
            pred = m.group(1)
    return actual, pred

# 4. Compute confusion matrices for each dataset
cms = []
for ds in dataset_names:
    with open(paths[ds], "r", encoding="utf-8") as f:
        data = json.load(f)
    y_true, y_pred = [], []
    for rec in data:
        a, p = extract_labels(rec)
        if a and p:
            y_true.append(a)
            y_pred.append(p)
    cms.append(confusion_matrix(y_true, y_pred, labels=label_order))

# 5. Stack rows: one block per actual label, across all datasets
n = len(dataset_names)
rows = []
for i in range(len(label_order)):
    for cm in cms:
        rows.append(cm[i])
arr = np.vstack(rows)  # shape = (6*n, 6)

# 6. Plot heatmap and save as PDF
fig, ax = plt.subplots(figsize=(10, 12))
im = ax.imshow(arr, aspect="auto")

# Colorbar
cbar = fig.colorbar(im, ax=ax)
cbar.set_label("Count", rotation=270, labelpad=15)

# Predicted labels on top
ax.set_xticks(np.arange(len(col_labels)))
ax.set_xticklabels(col_labels, fontsize=12)
ax.xaxis.set_label_position("top")
ax.xaxis.tick_top()
ax.set_xlabel("Predicted Label", fontsize=14, labelpad=10)

# Dataset names repeated for each actual-label block on the y-axis
ylabels = dataset_names * len(label_order)
ax.set_yticks(np.arange(len(ylabels)))
ax.set_yticklabels(ylabels, fontsize=10)
ax.set_ylabel("Actual Label", fontsize=14, labelpad=10)

# Horizontal separators between blocks
for idx in range(1, len(label_order)):
    sep = idx * n - 0.5
    ax.axhline(sep, color="white", linewidth=2)

# Annotate each cell
for i in range(arr.shape[0]):
    for j in range(arr.shape[1]):
        ax.text(j, i, int(arr[i, j]), ha="center", va="center", color="white")

# Block labels on left
for idx, lbl in enumerate(label_order):
    center = idx * n + (n - 1) / 2
    ax.text(-1.5, center, lbl, ha="center", va="center", fontsize=12)

plt.tight_layout()
output_pdf = "combined_confusion_matrix_lair_raw.pdf"
fig.savefig(output_pdf, format="pdf", bbox_inches="tight")
print(f"Saved combined confusion matrix to {output_pdf}")
